[Mohit Saharan](https://linkedin.com/in/msaharan), 20260511, P21, v2

Apache 2.0 License (see github.com/msaharan/dsaiengineering/LICENSE)

# Tactical Asset Allocation with TabPFN, TabICL, and XGBoost - 2

This notebook follows up on the [first tactical asset-allocation workflow](https://open.substack.com/pub/dsaiengineering/p/p20-tactical-asset-allocation-with-tabpfn-tabicl-xgboost?r=535odk&utm_campaign=post-expanded-share&utm_medium=web) by making the testbench more demanding. The previous version used a compact nine-ETF universe, static ticker identity features, a close-to-close diagnostic convention, and simple top-k portfolio mechanics. This version is designed to stress those assumptions.

The default workflow now uses a broader ETF universe across equity, sector, rates, credit, commodity, real-estate, and international exposures. It uses an identity-ablated feature policy by default, so static ticker and asset-group metadata are excluded from the model matrix unless the configuration is changed. It also supports a next-open-to-next-open execution-return convention and writes turnover-cap diagnostics alongside the existing transaction-cost sensitivity tables.

The objective remains narrow: given features available at a monthly signal date, score which assets are more likely to rank in the top group by next-month return within the configured universe. This is an educational research workflow, not investment advice and not a claim that any score is suitable for live trading without separate production-grade data, execution, risk, and compliance review.

The notebook writes reviewable CSV, TXT, and PNG artifacts to `tabpfn_tabicl_tactical_asset_allocation_20260511_outputs/` and creates a ZIP archive at the end. Use those files for post-run review because notebook editor output can be truncated.


## How to Run This Notebook on Kaggle

1. Import this notebook into Kaggle.
2. Enable GPU acceleration. The intended environment is Kaggle with two T4 GPUs.
3. Turn on Internet access. The notebook downloads packages, public market data, public macro data, and model checkpoints.
4. Add a Kaggle secret named `TABPFN_TOKEN` if local TabPFN weight access requires a token after accepting the Prior Labs license.
5. Optionally add `HF_TOKEN` for Hugging Face downloads. TabICL is usually downloadable without it, but a token can reduce rate-limit issues.

## 0. Imports, Secrets, and Configuration

In [1]:
%%time
# Kaggle / Colab setup.
# Run this cell once. If imports still fail, restart the notebook session and continue below.

import subprocess
import sys

PYPI_PACKAGES = [
    "xgboost>=2.0.0",
    "rich",
    "yfinance>=0.2.40",
    "tabpfn",
    "tabicl",
    "cupy-cuda12x",
    "tqdm",
    "scipy",
]

subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PYPI_PACKAGES], check=True)

import gc
import importlib.metadata as importlib_metadata
import json
import math
import os
from pathlib import Path
import platform
import subprocess
import time
import warnings
from traceback import format_exception_only
from urllib.parse import urlencode

import cupy as cp
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import sklearn
import torch
import yfinance as yf

from IPython.display import display
from rich.console import Console
from scipy.stats import loguniform, randint, spearmanr, uniform
from sklearn.base import clone
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss, log_loss, precision_recall_curve, roc_auc_score
from sklearn.model_selection import ParameterSampler
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="The `cv='prefit'` option is deprecated", category=FutureWarning)

console = Console()
SEED = 42
np.random.seed(SEED)

DATA_START_DATE = "2006-01-01"
DATA_END_DATE = "2026-05-12"  # yfinance end date is exclusive.
FRED_DOWNLOAD_START_DATE = "1990-01-01"

ASSET_TICKERS = {
    "SPY": "US large-cap equity",
    "QQQ": "US growth equity",
    "DIA": "US large-cap blue-chip equity",
    "IWM": "US small-cap equity",
    "EFA": "Developed ex-US equity",
    "EEM": "Emerging-market equity",
    "TLT": "Long-duration Treasury",
    "IEF": "Intermediate Treasury",
    "SHY": "Short-duration Treasury",
    "LQD": "Investment-grade credit",
    "HYG": "High-yield credit",
    "GLD": "Gold",
    "SLV": "Silver",
    "DBC": "Broad commodities",
    "VNQ": "US REIT",
    "IYR": "US real estate",
    "XLB": "US materials sector",
    "XLE": "US energy sector",
    "XLF": "US financials sector",
    "XLI": "US industrials sector",
    "XLK": "US technology sector",
    "XLP": "US consumer staples sector",
    "XLU": "US utilities sector",
    "XLV": "US healthcare sector",
    "XLY": "US consumer discretionary sector",
}
ASSET_GROUPS = {
    "SPY": "equity_us",
    "QQQ": "equity_us",
    "DIA": "equity_us",
    "IWM": "equity_us",
    "EFA": "equity_developed",
    "EEM": "equity_em",
    "TLT": "rates",
    "IEF": "rates",
    "SHY": "rates",
    "LQD": "credit",
    "HYG": "credit",
    "GLD": "commodity",
    "SLV": "commodity",
    "DBC": "commodity",
    "VNQ": "real_estate",
    "IYR": "real_estate",
    "XLB": "sector_us",
    "XLE": "sector_us",
    "XLF": "sector_us",
    "XLI": "sector_us",
    "XLK": "sector_us",
    "XLP": "sector_us",
    "XLU": "sector_us",
    "XLV": "sector_us",
    "XLY": "sector_us",
}
ASSET_RISK_BUCKET = {
    "SPY": 3,
    "QQQ": 4,
    "DIA": 3,
    "IWM": 4,
    "EFA": 4,
    "EEM": 4,
    "TLT": 2,
    "IEF": 1,
    "SHY": 1,
    "LQD": 2,
    "HYG": 3,
    "GLD": 3,
    "SLV": 4,
    "DBC": 4,
    "VNQ": 4,
    "IYR": 4,
    "XLB": 4,
    "XLE": 4,
    "XLF": 4,
    "XLI": 4,
    "XLK": 4,
    "XLP": 3,
    "XLU": 3,
    "XLV": 3,
    "XLY": 4,
}

FAST_MODE = False
if FAST_MODE:
    ASSET_TICKERS = {ticker: ASSET_TICKERS[ticker] for ticker in ["SPY", "QQQ", "IWM", "TLT", "IEF", "GLD", "EFA", "EEM", "XLE", "XLF"]}

PORTFOLIO_TOP_K = 5
if FAST_MODE:
    PORTFOLIO_TOP_K = 3
TARGET_TOP_K = min(PORTFOLIO_TOP_K, len(ASSET_TICKERS))
TRANSACTION_COST_BPS = 5.0
TRANSACTION_COST_SENSITIVITY_BPS = [0.0, 5.0, 10.0, 25.0, 50.0]
TURNOVER_CONSTRAINT_CAPS = [1.0, 0.5]
EXECUTION_RETURN_MODE = "next_open_to_next_open"  # options: close_to_close, next_open_to_next_open
FEATURE_SET_VARIANT = "identity_ablated"  # options: full, ticker_ablated, identity_ablated
MONTHS_PER_YEAR = 12

CONTEXT_END_DATE = "2011-12-31"
TUNING_END_DATE = "2017-12-31"
CALIBRATION_END_DATE = "2019-12-31"
HOLDOUT_START_DATE = "2020-01-01"

FEATURE_MAX_MISSING_RATE = 0.35
FEATURE_MIN_SELECTION_OBSERVATIONS = 36
RETURN_WINDOWS_DAYS = [21, 63, 126, 252]
VOL_WINDOWS_DAYS = [21, 63, 126]
DRAWDOWN_WINDOWS_DAYS = [63, 126, 252]
TREND_WINDOWS_DAYS = [63, 126, 200]
CROSS_ASSET_TICKERS_FOR_FEATURES = list(ASSET_TICKERS)

N_TFM_ESTIMATORS = 8
MEMORY_EFFICIENT_MODE = False
MEMORY_EFFICIENT_N_TFM_ESTIMATORS = 4
SAVE_FULL_PREDICTION_SCORES = True
XGBOOST_TUNING_ITERATIONS = 160
CLASSICAL_TUNING_CV_SPLITS = 6
MIN_CV_TRAIN_POSITIVES = 30
MIN_CV_VALIDATION_POSITIVES = 8
BOOTSTRAP_ITERATIONS = 400
BOOTSTRAP_CONFIDENCE_LEVEL = 0.95
PREDICTION_CHUNK_SIZE = 8192

FAST_MODE_N_TFM_ESTIMATORS = 2
FAST_MODE_XGBOOST_TUNING_ITERATIONS = 10
FAST_MODE_CLASSICAL_TUNING_CV_SPLITS = 2
FAST_MODE_BOOTSTRAP_ITERATIONS = 50
FAST_MODE_MIN_CV_TRAIN_POSITIVES = 8
FAST_MODE_MIN_CV_VALIDATION_POSITIVES = 2

if MEMORY_EFFICIENT_MODE and not FAST_MODE:
    N_TFM_ESTIMATORS = min(N_TFM_ESTIMATORS, MEMORY_EFFICIENT_N_TFM_ESTIMATORS)

if FAST_MODE:
    N_TFM_ESTIMATORS = min(N_TFM_ESTIMATORS, FAST_MODE_N_TFM_ESTIMATORS)
    XGBOOST_TUNING_ITERATIONS = min(XGBOOST_TUNING_ITERATIONS, FAST_MODE_XGBOOST_TUNING_ITERATIONS)
    CLASSICAL_TUNING_CV_SPLITS = min(CLASSICAL_TUNING_CV_SPLITS, FAST_MODE_CLASSICAL_TUNING_CV_SPLITS)
    BOOTSTRAP_ITERATIONS = min(BOOTSTRAP_ITERATIONS, FAST_MODE_BOOTSTRAP_ITERATIONS)
    MIN_CV_TRAIN_POSITIVES = min(MIN_CV_TRAIN_POSITIVES, FAST_MODE_MIN_CV_TRAIN_POSITIVES)
    MIN_CV_VALIDATION_POSITIVES = min(MIN_CV_VALIDATION_POSITIVES, FAST_MODE_MIN_CV_VALIDATION_POSITIVES)

RUN_GPU_XGBOOST = True
RUN_DIRECT_TABPFN = True
RUN_DIRECT_TABICL = True
RUN_LOGISTIC_REGRESSION_CPU_BENCHMARK = False
EVALUATE_CALIBRATION_VARIANTS = True

XGBOOST_TREE_METHOD = "hist"
XGBOOST_DEVICE = "cuda"
TABICL_CHECKPOINT_VERSION = "tabicl-classifier-v2-20260212.ckpt"
TABPFN_MODEL_NOTE = "default TabPFNClassifier checkpoint for the installed tabpfn package"

FRED_SERIES = {
    "DGS10": "treasury_10y_yield",
    "DGS2": "treasury_2y_yield",
    "T10Y2Y": "yield_curve_10y_2y",
    "BAMLH0A0HYM2": "high_yield_oas",
    "BAMLC0A0CM": "investment_grade_oas",
    "DFF": "fed_funds_rate",
    "DTB3": "t_bill_3m",
}
FRED_MIN_SELECTION_OBSERVATIONS = FEATURE_MIN_SELECTION_OBSERVATIONS
VIX_CBOE_CSV_URL = "https://cdn.cboe.com/api/global/us_indices/daily_prices/VIX_History.csv"

ARTIFACT_DIR = Path("tabpfn_tabicl_tactical_asset_allocation_20260511_outputs")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
artifact_manifest_rows = []
model_errors = []
model_rows = []
predictions = {}
prediction_frames = []
cuda_memory_snapshots = []

PUBLICATION_MODEL_ORDER = [
    "Rule[12M momentum top-k]",
    "Rule[6M momentum top-k]",
    "Rule[Low volatility top-k]",
    "XGBoost[GPU allocation scorer]",
    "TabPFN[Direct allocation scorer]",
    "TabICL[Direct allocation scorer]",
]
CALIBRATION_MODEL_ORDER = [
    "XGBoost[GPU allocation scorer] Calibration Base",
    "XGBoost[GPU allocation scorer] Calibrated",
    "TabPFN[Direct allocation scorer] Calibration Base",
    "TabICL[Direct allocation scorer] Calibration Base",
]


def register_artifact(path, kind, description=""):
    path = Path(path)
    artifact_manifest_rows.append(
        {
            "path": str(path),
            "filename": path.name,
            "kind": kind,
            "description": description,
            "exists": path.exists(),
            "size_bytes": path.stat().st_size if path.exists() else np.nan,
        }
    )
    return path


def save_artifact_table(table, filename, index=False, description=""):
    output_path = ARTIFACT_DIR / filename
    table.to_csv(output_path, index=index)
    register_artifact(output_path, "table", description)
    print(f"Saved {output_path}")
    return output_path


def save_text_artifact(filename, text, description=""):
    output_path = ARTIFACT_DIR / filename
    output_path.write_text(str(text), encoding="utf-8")
    register_artifact(output_path, "text", description)
    print(f"Saved {output_path}")
    return output_path


def installed_version(*package_names):
    for package_name in package_names:
        try:
            return importlib_metadata.version(package_name)
        except importlib_metadata.PackageNotFoundError:
            continue
    return "not installed"


def safe_string(value):
    if value is None:
        return ""
    if isinstance(value, (list, tuple)):
        return ", ".join(str(item) for item in value)
    return str(value)


def json_ready(value):
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, np.ndarray):
        return json_ready(value.tolist())
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    return value


def artifact_json(value):
    return json.dumps(json_ready(value), sort_keys=True)


def short_error(exc):
    return "".join(format_exception_only(type(exc), exc)).strip().replace("\n", " ")[:500]


def git_commit_or_unavailable(path):
    path = Path(path)
    if not path.exists():
        return "not available"
    try:
        result = subprocess.run(["git", "-C", str(path), "rev-parse", "HEAD"], capture_output=True, text=True, check=True, timeout=5)
        return result.stdout.strip()
    except Exception:
        return "not a git checkout or unavailable"


def source_provenance_frame():
    cwd = Path.cwd().resolve()
    candidates = {
        "notebook_working_directory": cwd,
        "parent_directory": cwd.parent,
        "tabpfn_sibling_repo": cwd.parent / "tabpfn",
        "tabicl_sibling_repo": cwd.parent / "tabicl",
    }
    return pd.DataFrame([{"Source": name, "Path": str(path), "Git Commit": git_commit_or_unavailable(path)} for name, path in candidates.items()])


def reset_cuda_peak_memory_stats():
    if not torch.cuda.is_available():
        return
    for device_index in range(torch.cuda.device_count()):
        torch.cuda.reset_peak_memory_stats(device_index)


def cuda_memory_snapshot(stage):
    if not torch.cuda.is_available():
        return pd.DataFrame()
    rows = []
    for device_index in range(torch.cuda.device_count()):
        free_bytes, total_bytes = torch.cuda.mem_get_info(device_index)
        rows.append(
            {
                "stage": stage,
                "device_index": device_index,
                "device_name": torch.cuda.get_device_name(device_index),
                "allocated_mb": torch.cuda.memory_allocated(device_index) / 1024**2,
                "reserved_mb": torch.cuda.memory_reserved(device_index) / 1024**2,
                "max_allocated_mb": torch.cuda.max_memory_allocated(device_index) / 1024**2,
                "free_mb": free_bytes / 1024**2,
                "total_mb": total_bytes / 1024**2,
            }
        )
    return pd.DataFrame(rows)


def record_cuda_memory(stage):
    snapshot = cuda_memory_snapshot(stage)
    if len(snapshot) > 0:
        cuda_memory_snapshots.append(snapshot)
    return snapshot


def cleanup_runtime_memory(stage="cleanup"):
    gc.collect()
    try:
        cp.get_default_memory_pool().free_all_blocks()
        cp.get_default_pinned_memory_pool().free_all_blocks()
    except Exception:
        pass
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        except Exception as exc:
            print(f"CUDA cleanup warning at {stage}: {short_error(exc)}")
    return record_cuda_memory(stage)


CUDA_DEVICE_COUNT = torch.cuda.device_count()
if CUDA_DEVICE_COUNT < 1:
    print("CUDA was not detected. Data construction can still run, but default model sections are designed for a GPU runtime.")
    XGBOOST_DEVICE = "cpu"
    RUN_DIRECT_TABPFN = False
    RUN_DIRECT_TABICL = False

TABICL_DEVICE = "cuda:0" if CUDA_DEVICE_COUNT else "cpu"
TABPFN_DEVICE = [f"cuda:{idx}" for idx in range(CUDA_DEVICE_COUNT)] if CUDA_DEVICE_COUNT else "cpu"


def load_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        value = UserSecretsClient().get_secret(name)
        if value:
            os.environ[name] = value
            return value
    except Exception:
        pass
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            os.environ[name] = value
            return value
    except Exception:
        pass
    return None


hf_token = load_secret("HF_TOKEN")
tabpfn_token = load_secret("TABPFN_TOKEN")
if not tabpfn_token:
    os.environ["TABPFN_NO_BROWSER"] = "1"
_ = os.environ.setdefault("TABPFN_DISABLE_TELEMETRY", "1")

environment_summary = pd.DataFrame(
    [
        {"Component": "Python", "Version": platform.python_version()},
        {"Component": "CUDA devices", "Version": CUDA_DEVICE_COUNT},
        {"Component": "pandas", "Version": installed_version("pandas")},
        {"Component": "numpy", "Version": installed_version("numpy")},
        {"Component": "scikit-learn", "Version": sklearn.__version__},
        {"Component": "xgboost", "Version": installed_version("xgboost")},
        {"Component": "torch", "Version": torch.__version__},
        {"Component": "cupy", "Version": cp.__version__},
        {"Component": "yfinance", "Version": installed_version("yfinance")},
        {"Component": "tabpfn", "Version": installed_version("tabpfn")},
        {"Component": "tabicl", "Version": installed_version("tabicl")},
    ]
)

display(environment_summary)
save_artifact_table(environment_summary, "environment_summary.csv")
source_provenance_summary = source_provenance_frame()
display(source_provenance_summary)
save_artifact_table(source_provenance_summary, "source_provenance_summary.csv")
initial_cuda_memory = record_cuda_memory("after_imports_and_configuration")
if len(initial_cuda_memory) > 0:
    display(initial_cuda_memory.round(2))
    save_artifact_table(initial_cuda_memory, "cuda_memory_after_imports.csv")

configuration_summary = pd.DataFrame(
    [
        {"Parameter": "DATA_START_DATE", "Value": DATA_START_DATE},
        {"Parameter": "DATA_END_DATE", "Value": DATA_END_DATE},
        {"Parameter": "FRED_DOWNLOAD_START_DATE", "Value": FRED_DOWNLOAD_START_DATE},
        {"Parameter": "FRED_MIN_SELECTION_OBSERVATIONS", "Value": FRED_MIN_SELECTION_OBSERVATIONS},
        {"Parameter": "ASSET_TICKERS", "Value": safe_string(list(ASSET_TICKERS))},
        {"Parameter": "ASSET_COUNT", "Value": len(ASSET_TICKERS)},
        {"Parameter": "TARGET_TOP_K", "Value": TARGET_TOP_K},
        {"Parameter": "PORTFOLIO_TOP_K", "Value": PORTFOLIO_TOP_K},
        {"Parameter": "TRANSACTION_COST_BPS", "Value": TRANSACTION_COST_BPS},
        {"Parameter": "TRANSACTION_COST_SENSITIVITY_BPS", "Value": safe_string(TRANSACTION_COST_SENSITIVITY_BPS)},
        {"Parameter": "TURNOVER_CONSTRAINT_CAPS", "Value": safe_string(TURNOVER_CONSTRAINT_CAPS)},
        {"Parameter": "EXECUTION_RETURN_MODE", "Value": EXECUTION_RETURN_MODE},
        {"Parameter": "FEATURE_SET_VARIANT", "Value": FEATURE_SET_VARIANT},
        {"Parameter": "CONTEXT_END_DATE", "Value": CONTEXT_END_DATE},
        {"Parameter": "TUNING_END_DATE", "Value": TUNING_END_DATE},
        {"Parameter": "CALIBRATION_END_DATE", "Value": CALIBRATION_END_DATE},
        {"Parameter": "HOLDOUT_START_DATE", "Value": HOLDOUT_START_DATE},
        {"Parameter": "FAST_MODE", "Value": FAST_MODE},
        {"Parameter": "MEMORY_EFFICIENT_MODE", "Value": MEMORY_EFFICIENT_MODE},
        {"Parameter": "MEMORY_EFFICIENT_N_TFM_ESTIMATORS", "Value": MEMORY_EFFICIENT_N_TFM_ESTIMATORS},
        {"Parameter": "SAVE_FULL_PREDICTION_SCORES", "Value": SAVE_FULL_PREDICTION_SCORES},
        {"Parameter": "N_TFM_ESTIMATORS", "Value": N_TFM_ESTIMATORS},
        {"Parameter": "RUN_GPU_XGBOOST", "Value": RUN_GPU_XGBOOST},
        {"Parameter": "RUN_DIRECT_TABPFN", "Value": RUN_DIRECT_TABPFN},
        {"Parameter": "RUN_DIRECT_TABICL", "Value": RUN_DIRECT_TABICL},
        {"Parameter": "RUN_LOGISTIC_REGRESSION_CPU_BENCHMARK", "Value": RUN_LOGISTIC_REGRESSION_CPU_BENCHMARK},
        {"Parameter": "XGBOOST_TUNING_ITERATIONS", "Value": XGBOOST_TUNING_ITERATIONS},
        {"Parameter": "CLASSICAL_TUNING_CV_SPLITS", "Value": CLASSICAL_TUNING_CV_SPLITS},
        {"Parameter": "BOOTSTRAP_ITERATIONS", "Value": BOOTSTRAP_ITERATIONS},
        {"Parameter": "XGBOOST_DEVICE", "Value": XGBOOST_DEVICE},
        {"Parameter": "TABPFN_DEVICE", "Value": safe_string(TABPFN_DEVICE)},
        {"Parameter": "TABICL_DEVICE", "Value": TABICL_DEVICE},
    ]
)
display(configuration_summary)
save_artifact_table(configuration_summary, "configuration_summary.csv", description="Notebook configuration values used for this run.")

method_contract = f"""
# Tactical Asset-Allocation Workflow Contract

Universe: {safe_string(list(ASSET_TICKERS))}
Task: score each asset at monthly signal date t for whether it will be in the top {TARGET_TOP_K} assets by next-month total return inside the configured universe.
Target construction: next-month asset return is computed from the configured execution-return convention (`{EXECUTION_RETURN_MODE}`); close-to-close and next-open diagnostic returns are both retained for audit. The top-k label is assigned only within the same signal month.
Feature availability policy: features use data available at or before the monthly signal date; forward returns, forward excess returns, and top-k labels are excluded from model features. The configured feature variant is `{FEATURE_SET_VARIANT}`; identity-ablated mode removes static ticker, group, and risk-bucket metadata from the model matrix. FRED series are included in model features only when they have enough non-missing observations in the model-selection window; unavailable optional credit-spread histories are recorded but not imputed into the training set.
Splits: context through {CONTEXT_END_DATE}; model-selection history through {TUNING_END_DATE}; calibration through {CALIBRATION_END_DATE}; holdout from {HOLDOUT_START_DATE}.
Validation: rolling-origin chronological cross-validation on monthly groups with positive-count checks.
Default model families: deterministic allocation rules, GPU XGBoost, direct TabPFN, and direct TabICL. CPU-only Logistic Regression is available but disabled by default.
Execution convention: scores are formed after month-end close data is available and are evaluated with `{EXECUTION_RETURN_MODE}`. The default next-open convention uses the first adjusted open of the next month through the first adjusted open of the following month. The diagnostic still does not model intraday slippage, taxes, market impact, or mandate constraints.
Portfolio diagnostic: monthly top-k equal-weight allocation from model scores, compared with equal-weight, SPY-only, 60/40 SPY/TLT, inverse-volatility, and deterministic momentum rules. The main portfolio table uses {TRANSACTION_COST_BPS:.1f} basis points per unit of one-way turnover, with separate transaction-cost and turnover-cap sensitivity artifacts for stress review.
Interpretation: educational workflow test; not investment advice; not a claim of market predictability or deployable trading performance.
""".strip()
save_text_artifact("method_contract.md", method_contract, description="Plain-language modeling and evaluation contract.")

print(f"Run mode: {'FAST smoke test' if FAST_MODE else 'full research run'}")
print(f"Asset universe: {safe_string(list(ASSET_TICKERS))}")
print(f"Splits: context <= {CONTEXT_END_DATE}; selection <= {TUNING_END_DATE}; calibration <= {CALIBRATION_END_DATE}; holdout >= {HOLDOUT_START_DATE}")
print(f"HF_TOKEN found: {bool(hf_token)}; TABPFN_TOKEN found: {bool(tabpfn_token)}")
print(f"TabPFN device: {TABPFN_DEVICE}; TabICL device: {TABICL_DEVICE}; XGBoost device: {XGBOOST_DEVICE}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 660.6/660.6 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 252.9/252.9 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.2/240.2 kB 18.6 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.


,Component,Version
0,Python,3.12.12
1,CUDA devices,2
2,pandas,2.3.3
3,numpy,2.0.2
4,scikit-learn,1.6.1
5,xgboost,3.2.0
6,torch,2.10.0+cu128
7,cupy,14.0.1
8,yfinance,0.2.66
9,tabpfn,7.1.1


Saved tabpfn_tabicl_tactical_asset_allocation_20260511_outputs/environment_summary.csv


,Source,Path,Git Commit
0,notebook_working_directory,/kaggle/working,not a git checkout or unavailable
1,parent_directory,/kaggle,not a git checkout or unavailable
2,tabpfn_sibling_repo,/kaggle/tabpfn,not available
3,tabicl_sibling_repo,/kaggle/tabicl,not available


Saved tabpfn_tabicl_tactical_asset_allocation_20260511_outputs/source_provenance_summary.csv


,stage,device_index,device_name,allocated_mb,reserved_mb,max_allocated_mb,free_mb,total_mb
0,after_imports_and_configuration,0,Tesla T4,0.0,0.0,0.0,14807.81,14912.69
1,after_imports_and_configuration,1,Tesla T4,0.0,0.0,0.0,14807.81,14912.69


Saved tabpfn_tabicl_tactical_asset_allocation_20260511_outputs/cuda_memory_after_imports.csv


,Parameter,Value
0,DATA_START_DATE,2006-01-01
1,DATA_END_DATE,2026-05-12
2,FRED_DOWNLOAD_START_DATE,1990-01-01
3,FRED_MIN_SELECTION_OBSERVATIONS,36
4,ASSET_TICKERS,"SPY, QQQ, DIA, IWM, EFA, EEM, TLT, IEF, SHY, L..."
5,ASSET_COUNT,25
6,TARGET_TOP_K,5
7,PORTFOLIO_TOP_K,5
8,TRANSACTION_COST_BPS,5.0
9,TRANSACTION_COST_SENSITIVITY_BPS,"0.0, 5.0, 10.0, 25.0, 50.0"


Saved tabpfn_tabicl_tactical_asset_allocation_20260511_outputs/configuration_summary.csv
Saved tabpfn_tabicl_tactical_asset_allocation_20260511_outputs/method_contract.md
Run mode: full research run
Asset universe: SPY, QQQ, DIA, IWM, EFA, EEM, TLT, IEF, SHY, LQD, HYG, GLD, SLV, DBC, VNQ, IYR, XLB, XLE, XLF, XLI, XLK, XLP, XLU, XLV, XLY
Splits: context <= 2011-12-31; selection <= 2017-12-31; calibration <= 2019-12-31; holdout >= 2020-01-01
HF_TOKEN found: True; TABPFN_TOKEN found: True
TabPFN device: ['cuda:0', 'cuda:1']; TabICL device: cuda:0; XGBoost device: cuda
CPU times: user 4.23 s, sys: 1.4 s, total: 5.63 s
Wall time: 14.7 s


## 1. Load Public Market, Volatility, and Macro Data

The workflow uses public adjusted ETF prices, Cboe VIX history, and FRED macro series. Some FRED-hosted credit-spread series can be restricted to recent public history; the notebook records each FRED series availability and includes a series in model features only when enough model-selection-window history is present. Public data improves reproducibility, but it is not a substitute for a point-in-time institutional data system. The leakage checklist later records the assumptions that follow from this data choice.

In [2]:
def normalize_date_index(frame, date_column=None):
    out = frame.copy()
    if date_column is not None:
        out[date_column] = pd.to_datetime(out[date_column])
        out = out.set_index(date_column)
    out.index = pd.to_datetime(out.index).tz_localize(None)
    return out.sort_index()


def download_yfinance_ohlcv(ticker):
    frame = yf.download(ticker, start=DATA_START_DATE, end=DATA_END_DATE, auto_adjust=False, progress=False, threads=False)
    if frame.empty:
        raise ValueError(f"yfinance returned no rows for {ticker}.")
    if isinstance(frame.columns, pd.MultiIndex):
        frame.columns = [column[0] for column in frame.columns]
    frame = normalize_date_index(frame)
    required = {"Open", "High", "Low", "Close", "Volume"}
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"{ticker} data missing required columns: {sorted(missing)}")
    if "Adj Close" not in frame.columns:
        frame["Adj Close"] = frame["Close"]
    return frame[["Open", "High", "Low", "Close", "Adj Close", "Volume"]].copy()


price_frames = {}
price_source_rows = []
for ticker in tqdm(list(ASSET_TICKERS), desc="Downloading ETF OHLCV", unit="ticker"):
    frame = download_yfinance_ohlcv(ticker)
    price_frames[ticker] = frame
    price_source_rows.append(
        {
            "source": "yfinance",
            "symbol": ticker,
            "rows": len(frame),
            "start": frame.index.min().date().isoformat(),
            "end": frame.index.max().date().isoformat(),
            "description": ASSET_TICKERS[ticker],
        }
    )

adj_close = pd.concat({ticker: frame["Adj Close"] for ticker, frame in price_frames.items()}, axis=1).sort_index()
volume = pd.concat({ticker: frame["Volume"] for ticker, frame in price_frames.items()}, axis=1).sort_index()
open_px = pd.concat({ticker: frame["Open"] for ticker, frame in price_frames.items()}, axis=1).sort_index()
high_px = pd.concat({ticker: frame["High"] for ticker, frame in price_frames.items()}, axis=1).sort_index()
low_px = pd.concat({ticker: frame["Low"] for ticker, frame in price_frames.items()}, axis=1).sort_index()
close_px = pd.concat({ticker: frame["Close"] for ticker, frame in price_frames.items()}, axis=1).sort_index()


def load_vix_history():
    vix = pd.read_csv(VIX_CBOE_CSV_URL)
    date_column = "DATE" if "DATE" in vix.columns else vix.columns[0]
    vix = normalize_date_index(vix, date_column=date_column)
    rename_map = {column: f"vix_{column.lower()}" for column in vix.columns}
    vix = vix.rename(columns=rename_map)
    close_candidates = [column for column in vix.columns if "close" in column.lower()]
    if close_candidates and "vix_close" not in vix.columns:
        vix = vix.rename(columns={close_candidates[0]: "vix_close"})
    return vix


def load_fred_series(series_id, output_name):
    query = urlencode(
        {
            "id": series_id,
            "cosd": FRED_DOWNLOAD_START_DATE,
            "coed": DATA_END_DATE,
            "observation_start": FRED_DOWNLOAD_START_DATE,
        }
    )
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?{query}"
    frame = pd.read_csv(url)
    date_column = "observation_date" if "observation_date" in frame.columns else frame.columns[0]
    frame = normalize_date_index(frame, date_column=date_column)
    value_column = [column for column in frame.columns if column != date_column][0]
    values = pd.to_numeric(frame[value_column].replace(".", np.nan), errors="coerce")
    return values.rename(output_name).to_frame()

vix_history = load_vix_history()
price_source_rows.append(
    {
        "source": "Cboe VIX CSV",
        "symbol": "VIX",
        "rows": len(vix_history),
        "start": vix_history.index.min().date().isoformat(),
        "end": vix_history.index.max().date().isoformat(),
        "description": "Cboe VIX daily history",
    }
)

fred_frames = []
fred_availability_rows = []
selection_cutoff = pd.Timestamp(TUNING_END_DATE)
preholdout_cutoff = pd.Timestamp(CALIBRATION_END_DATE)
for series_id, output_name in tqdm(FRED_SERIES.items(), desc="Downloading FRED series", unit="series"):
    try:
        frame = load_fred_series(series_id, output_name)
        start_value = frame.index.min().date().isoformat() if len(frame) else ""
        end_value = frame.index.max().date().isoformat() if len(frame) else ""
        price_source_rows.append(
            {
                "source": "FRED CSV",
                "symbol": series_id,
                "rows": len(frame),
                "start": start_value,
                "end": end_value,
                "description": output_name,
            }
        )
        selection_non_null = int(frame.loc[frame.index <= selection_cutoff, output_name].notna().sum()) if len(frame) else 0
        preholdout_non_null = int(frame.loc[frame.index <= preholdout_cutoff, output_name].notna().sum()) if len(frame) else 0
        include_in_features = selection_non_null >= FRED_MIN_SELECTION_OBSERVATIONS
        status = "included" if include_in_features else "excluded_insufficient_selection_history"
        fred_availability_rows.append(
            {
                "series_id": series_id,
                "feature_name": output_name,
                "rows": len(frame),
                "start": start_value,
                "end": end_value,
                "selection_non_null": selection_non_null,
                "preholdout_non_null": preholdout_non_null,
                "minimum_selection_observations": FRED_MIN_SELECTION_OBSERVATIONS,
                "include_in_features": include_in_features,
                "status": status,
            }
        )
        if include_in_features:
            fred_frames.append(frame)
        else:
            print(f"FRED series {series_id} excluded from model features: only {selection_non_null} non-null selection-window observations.")
    except Exception as exc:
        model_errors.append({"Model": f"FRED[{series_id}]", "Error": short_error(exc)})
        fred_availability_rows.append(
            {
                "series_id": series_id,
                "feature_name": output_name,
                "rows": 0,
                "start": "",
                "end": "",
                "selection_non_null": 0,
                "preholdout_non_null": 0,
                "minimum_selection_observations": FRED_MIN_SELECTION_OBSERVATIONS,
                "include_in_features": False,
                "status": f"load_failed: {short_error(exc)}",
            }
        )
        print(f"Could not load FRED series {series_id}: {short_error(exc)}")

fred_series_availability_summary = pd.DataFrame(fred_availability_rows)
if len(fred_series_availability_summary) > 0:
    display(fred_series_availability_summary)
    save_artifact_table(fred_series_availability_summary, "fred_series_availability_summary.csv", description="FRED source coverage and feature-inclusion decisions.")

macro_daily = pd.concat(fred_frames, axis=1).sort_index() if fred_frames else pd.DataFrame(index=adj_close.index)
macro_daily = macro_daily.reindex(adj_close.index).ffill()
vix_daily = vix_history.reindex(adj_close.index).ffill()

source_summary = pd.DataFrame(price_source_rows)
display(source_summary)
save_artifact_table(source_summary, "data_source_summary.csv", description="External data sources used by the notebook.")

price_coverage = pd.DataFrame(
    {
        "ticker": adj_close.columns,
        "first_valid_date": [adj_close[column].first_valid_index().date().isoformat() for column in adj_close.columns],
        "last_valid_date": [adj_close[column].last_valid_index().date().isoformat() for column in adj_close.columns],
        "missing_rate": [float(adj_close[column].isna().mean()) for column in adj_close.columns],
    }
)
display(price_coverage)
save_artifact_table(price_coverage, "price_coverage_summary.csv")


FRED series BAMLH0A0HYM2 excluded from model features: only 0 non-null selection-window observations.
FRED series BAMLC0A0CM excluded from model features: only 0 non-null selection-window observations.


,series_id,feature_name,rows,start,end,selection_non_null,preholdout_non_null,minimum_selection_observations,include_in_features,status
0,DGS10,treasury_10y_yield,9483,1990-01-02,2026-05-07,7006,7505,36,True,included
1,DGS2,treasury_2y_yield,9483,1990-01-02,2026-05-07,7006,7505,36,True,included
2,T10Y2Y,yield_curve_10y_2y,9484,1990-01-02,2026-05-08,7006,7505,36,True,included
3,BAMLH0A0HYM2,high_yield_oas,794,2023-05-09,2026-05-07,0,0,36,False,excluded_insufficient_selection_history
4,BAMLC0A0CM,investment_grade_oas,794,2023-05-09,2026-05-07,0,0,36,False,excluded_insufficient_selection_history
5,DFF,fed_funds_rate,13276,1990-01-01,2026-05-07,10227,10957,36,True,included
6,DTB3,t_bill_3m,9483,1990-01-02,2026-05-07,7006,7505,36,True,included


Saved tabpfn_tabicl_tactical_asset_allocation_20260511_outputs/fred_series_availability_summary.csv


,source,symbol,rows,start,end,description
0,yfinance,SPY,5119,2006-01-03,2026-05-08,US large-cap equity
1,yfinance,QQQ,5119,2006-01-03,2026-05-08,US growth equity
2,yfinance,DIA,5119,2006-01-03,2026-05-08,US large-cap blue-chip equity
3,yfinance,IWM,5119,2006-01-03,2026-05-08,US small-cap equity
4,yfinance,EFA,5119,2006-01-03,2026-05-08,Developed ex-US equity
5,yfinance,EEM,5119,2006-01-03,2026-05-08,Emerging-market equity
6,yfinance,TLT,5119,2006-01-03,2026-05-08,Long-duration Treasury
7,yfinance,IEF,5119,2006-01-03,2026-05-08,Intermediate Treasury
8,yfinance,SHY,5119,2006-01-03,2026-05-08,Short-duration Treasury
9,yfinance,LQD,5119,2006-01-03,2026-05-08,Investment-grade credit


Saved tabpfn_tabicl_tactical_asset_allocation_20260511_outputs/data_source_summary.csv


,ticker,first_valid_date,last_valid_date,missing_rate
0,SPY,2006-01-03,2026-05-08,0.000000
1,QQQ,2006-01-03,2026-05-08,0.000000
2,DIA,2006-01-03,2026-05-08,0.000000
3,IWM,2006-01-03,2026-05-08,0.000000
4,EFA,2006-01-03,2026-05-08,0.000000
5,EEM,2006-01-03,2026-05-08,0.000000
6,TLT,2006-01-03,2026-05-08,0.000000
7,IEF,2006-01-03,2026-05-08,0.000000
8,SHY,2006-01-03,2026-05-08,0.000000
9,LQD,2006-01-03,2026-05-08,0.000000


Saved tabpfn_tabicl_tactical_asset_allocation_20260511_outputs/price_coverage_summary.csv


PosixPath('tabpfn_tabicl_tactical_asset_allocation_20260511_outputs/price_coverage_summary.csv')

## 2. Build a Point-in-Time Monthly Allocation Table

The modeling table has one row per asset and monthly signal date. Features are computed from information available at or before the signal date. The signal date is the month-end close used for the public-data diagnostic; scores should be interpreted as being formed after those close observations are available and then evaluated on the following close-to-close monthly return. The default target is whether the asset is in the top `TARGET_TOP_K` assets for the next month.

In [3]:
def annualized_realized_vol(log_returns, window):
    return log_returns.rolling(window).std() * math.sqrt(252)


def downside_realized_vol(log_returns, window):
    downside = log_returns.where(log_returns < 0.0, 0.0)
    return downside.rolling(window).std() * math.sqrt(252)


def rolling_drawdown(price, window):
    return price / price.rolling(window).max() - 1.0


def rolling_zscore(series, window):
    rolling_mean = series.rolling(window).mean()
    rolling_std = series.rolling(window).std()
    return (series - rolling_mean) / rolling_std.replace(0.0, np.nan)


feature_series = {}
adj_close_aligned = adj_close.ffill()
volume_aligned = volume.reindex(adj_close_aligned.index).ffill()
asset_log_returns = {}
asset_simple_returns = {}

for ticker in tqdm(adj_close_aligned.columns, desc="Building daily asset features", unit="ticker"):
    price = adj_close_aligned[ticker]
    log_return = np.log(price).diff()
    simple_return = price.pct_change()
    asset_log_returns[ticker] = log_return
    asset_simple_returns[ticker] = simple_return

    for window in RETURN_WINDOWS_DAYS:
        feature_series[f"{ticker.lower()}_return_{window}d"] = price.pct_change(window)
    for window in VOL_WINDOWS_DAYS:
        feature_series[f"{ticker.lower()}_rv_{window}d"] = annualized_realized_vol(log_return, window)
        feature_series[f"{ticker.lower()}_downside_rv_{window}d"] = downside_realized_vol(log_return, window)
    for window in DRAWDOWN_WINDOWS_DAYS:
        feature_series[f"{ticker.lower()}_drawdown_{window}d"] = rolling_drawdown(price, window)
    for window in TREND_WINDOWS_DAYS:
        feature_series[f"{ticker.lower()}_ma_distance_{window}d"] = price / price.rolling(window).mean() - 1.0
    feature_series[f"{ticker.lower()}_volume_zscore_63d"] = rolling_zscore(np.log1p(volume_aligned[ticker]), 63)

# Cross-asset relationships and regime features.
spy_return = asset_simple_returns.get("SPY")
if spy_return is not None:
    for ticker in adj_close_aligned.columns:
        if ticker == "SPY":
            continue
        joined = pd.concat([asset_simple_returns[ticker], spy_return], axis=1).dropna()
        rolling_cov = joined.iloc[:, 0].rolling(126).cov(joined.iloc[:, 1])
        rolling_var = joined.iloc[:, 1].rolling(126).var()
        feature_series[f"{ticker.lower()}_beta_to_spy_126d"] = (rolling_cov / rolling_var.replace(0.0, np.nan)).reindex(adj_close_aligned.index)

if "vix_close" in vix_daily.columns:
    feature_series["vix_close"] = vix_daily["vix_close"]
    feature_series["vix_change_21d"] = vix_daily["vix_close"].diff(21)
    feature_series["vix_zscore_252d"] = rolling_zscore(vix_daily["vix_close"], 252)

for column in macro_daily.columns:
    series = macro_daily[column]
    feature_series[column] = series
    feature_series[f"{column}_change_21d"] = series.diff(21)
    feature_series[f"{column}_zscore_252d"] = rolling_zscore(series, 252)

market_feature_daily = pd.DataFrame(feature_series).sort_index()
monthly_market_features = market_feature_daily.resample("M").last()
monthly_prices = adj_close_aligned.resample("M").last()
adjustment_ratio = (adj_close_aligned / close_px.replace(0.0, np.nan)).replace([np.inf, -np.inf], np.nan)
adj_open_px = (open_px * adjustment_ratio).reindex(adj_close_aligned.index).ffill()
monthly_open_prices = adj_open_px.resample("M").first()
monthly_open_prices_for_returns = monthly_open_prices.copy()
# The yfinance end date is exclusive. If it falls inside the latest resampled month,
# drop that partial month as a signal month. Keep its first adjusted open for next-open
# exit-return measurement because that open is already known after the month begins.
requested_end_period = pd.Timestamp(DATA_END_DATE).to_period("M")
if len(monthly_prices) > 0 and monthly_prices.index.max().to_period("M") == requested_end_period:
    monthly_prices = monthly_prices.iloc[:-1].copy()
    monthly_market_features = monthly_market_features.loc[monthly_market_features.index.isin(monthly_prices.index)].copy()
monthly_close_to_close_forward_returns = monthly_prices.shift(-1) / monthly_prices - 1.0
monthly_next_open_forward_returns = monthly_open_prices_for_returns.shift(-2) / monthly_open_prices_for_returns.shift(-1) - 1.0
if EXECUTION_RETURN_MODE == "close_to_close":
    monthly_forward_returns = monthly_close_to_close_forward_returns
elif EXECUTION_RETURN_MODE == "next_open_to_next_open":
    monthly_forward_returns = monthly_next_open_forward_returns
else:
    raise ValueError(f"Unsupported EXECUTION_RETURN_MODE: {EXECUTION_RETURN_MODE}")
monthly_asset_returns = monthly_prices.pct_change()

asset_metadata_rows = []
for ticker in ASSET_TICKERS:
    row = {
        "asset": ticker,
        "asset_risk_bucket": ASSET_RISK_BUCKET.get(ticker, np.nan),
    }
    for asset_ticker in ASSET_TICKERS:
        row[f"asset_is_{asset_ticker.lower()}"] = int(ticker == asset_ticker)
    for group in sorted(set(ASSET_GROUPS.values())):
        row[f"asset_group_{group}"] = int(ASSET_GROUPS.get(ticker) == group)
    asset_metadata_rows.append(row)
asset_metadata = pd.DataFrame(asset_metadata_rows).set_index("asset")

panel_rows = []
for signal_date in tqdm(monthly_market_features.index, desc="Building monthly panel", unit="month"):
    market_row = monthly_market_features.loc[signal_date]
    if signal_date not in monthly_forward_returns.index:
        continue
    forward_row = monthly_forward_returns.loc[signal_date]
    close_to_close_row = monthly_close_to_close_forward_returns.loc[signal_date] if signal_date in monthly_close_to_close_forward_returns.index else pd.Series(dtype=float)
    next_open_row = monthly_next_open_forward_returns.loc[signal_date] if signal_date in monthly_next_open_forward_returns.index else pd.Series(dtype=float)
    same_month_returns = monthly_asset_returns.loc[signal_date] if signal_date in monthly_asset_returns.index else pd.Series(dtype=float)
    available_forward = forward_row.dropna()
    if len(available_forward) < max(3, TARGET_TOP_K):
        continue
    top_assets = set(available_forward.sort_values(ascending=False).head(TARGET_TOP_K).index)
    universe_forward_mean = float(available_forward.mean())
    universe_forward_rank = available_forward.rank(ascending=False, method="first")
    for ticker, forward_return in available_forward.items():
        row = {
            "date": pd.Timestamp(signal_date),
            "asset": ticker,
            "target_top_k_next_1m": int(ticker in top_assets),
            "forward_1m_return": float(forward_return),
            "forward_1m_close_to_close_return": float(close_to_close_row.get(ticker, np.nan)),
            "forward_1m_next_open_return": float(next_open_row.get(ticker, np.nan)),
            "forward_1m_excess_return": float(forward_return - universe_forward_mean),
            "forward_1m_rank": float(universe_forward_rank[ticker]),
            "universe_forward_1m_return": universe_forward_mean,
            "execution_return_mode": EXECUTION_RETURN_MODE,
            "asset_month_return": float(same_month_returns.get(ticker, np.nan)),
        }
        row.update(asset_metadata.loc[ticker].to_dict())
        lower = ticker.lower()
        row["asset_return_21d"] = market_row.get(f"{lower}_return_21d", np.nan)
        row["asset_return_63d"] = market_row.get(f"{lower}_return_63d", np.nan)
        row["asset_return_126d"] = market_row.get(f"{lower}_return_126d", np.nan)
        row["asset_return_252d"] = market_row.get(f"{lower}_return_252d", np.nan)
        row["asset_rv_21d"] = market_row.get(f"{lower}_rv_21d", np.nan)
        row["asset_rv_63d"] = market_row.get(f"{lower}_rv_63d", np.nan)
        row["asset_rv_126d"] = market_row.get(f"{lower}_rv_126d", np.nan)
        row["asset_downside_rv_63d"] = market_row.get(f"{lower}_downside_rv_63d", np.nan)
        row["asset_drawdown_126d"] = market_row.get(f"{lower}_drawdown_126d", np.nan)
        row["asset_drawdown_252d"] = market_row.get(f"{lower}_drawdown_252d", np.nan)
        row["asset_ma_distance_126d"] = market_row.get(f"{lower}_ma_distance_126d", np.nan)
        row["asset_volume_zscore_63d"] = market_row.get(f"{lower}_volume_zscore_63d", np.nan)
        row["asset_beta_to_spy_126d"] = market_row.get(f"{lower}_beta_to_spy_126d", 1.0 if ticker == "SPY" else np.nan)
        for feature_name, feature_value in market_row.items():
            if feature_name.startswith(f"{lower}_"):
                continue
            row[f"market_{feature_name}"] = feature_value
        panel_rows.append(row)

model_frame = pd.DataFrame(panel_rows).sort_values(["date", "asset"]).reset_index(drop=True)
model_frame = model_frame.replace([np.inf, -np.inf], np.nan)
model_frame = model_frame.dropna(subset=["target_top_k_next_1m", "forward_1m_return"]).reset_index(drop=True)
model_frame["target_top_k_next_1m"] = model_frame["target_top_k_next_1m"].astype(int)
model_frame["month"] = model_frame["date"].dt.to_period("M").astype(str)
model_frame["month_index"] = pd.factorize(model_frame["month"])[0]

if FAST_MODE:
    # Keep the full chronology but reduce the number of rows by using the configured smaller universe.
    model_frame = model_frame.loc[model_frame["asset"].isin(ASSET_TICKERS)].reset_index(drop=True)

split_preview = model_frame.groupby("month").agg(rows=("asset", "size"), positive_rows=("target_top_k_next_1m", "sum"), mean_forward_return=("forward_1m_return", "mean")).reset_index()
model_frame_head = model_frame.head(30)

display(model_frame_head)
display(split_preview.tail(12))
save_artifact_table(model_frame_head, "model_frame_head.csv")
save_artifact_table(split_preview, "monthly_target_preview.csv")

target_definition_summary = pd.DataFrame(
    [
        {"metric": "rows", "value": len(model_frame)},
        {"metric": "months", "value": model_frame["month"].nunique()},
        {"metric": "assets", "value": model_frame["asset"].nunique()},
        {"metric": "positive_rows", "value": int(model_frame["target_top_k_next_1m"].sum())},
        {"metric": "positive_rate", "value": float(model_frame["target_top_k_next_1m"].mean())},
        {"metric": "target_top_k", "value": TARGET_TOP_K},
    ]
)
display(target_definition_summary)
save_artifact_table(target_definition_summary, "target_definition_summary.csv")

execution_return_summary = pd.DataFrame(
    [
        {
            "return_mode": "close_to_close",
            "non_null_rows": int(monthly_close_to_close_forward_returns.stack(dropna=True).shape[0]),
            "first_signal_month": str(monthly_close_to_close_forward_returns.dropna(how="all").index.min().to_period("M")) if len(monthly_close_to_close_forward_returns.dropna(how="all")) else "",
            "last_signal_month": str(monthly_close_to_close_forward_returns.dropna(how="all").index.max().to_period("M")) if len(monthly_close_to_close_forward_returns.dropna(how="all")) else "",
            "is_selected_target_mode": EXECUTION_RETURN_MODE == "close_to_close",
        },
        {
            "return_mode": "next_open_to_next_open",
            "non_null_rows": int(monthly_next_open_forward_returns.stack(dropna=True).shape[0]),
            "first_signal_month": str(monthly_next_open_forward_returns.dropna(how="all").index.min().to_period("M")) if len(monthly_next_open_forward_returns.dropna(how="all")) else "",
            "last_signal_month": str(monthly_next_open_forward_returns.dropna(how="all").index.max().to_period("M")) if len(monthly_next_open_forward_returns.dropna(how="all")) else "",
            "is_selected_target_mode": EXECUTION_RETURN_MODE == "next_open_to_next_open",
        },
    ]
)
display(execution_return_summary)
save_artifact_table(execution_return_summary, "execution_return_summary.csv", description="Return conventions available for target and portfolio diagnostics.")


Building daily asset features:   0%|          | 0/25 [00:00<?, ?ticker/s]

Building monthly panel:   0%|          | 0/244 [00:00<?, ?month/s]

,date,asset,target_top_k_next_1m,forward_1m_return,forward_1m_close_to_close_return,forward_1m_next_open_return,forward_1m_excess_return,forward_1m_rank,universe_forward_1m_return,execution_return_mode,...,market_spy_downside_rv_126d,market_spy_drawdown_63d,market_spy_drawdown_126d,market_spy_drawdown_252d,market_spy_ma_distance_63d,market_spy_ma_distance_126d,market_spy_ma_distance_200d,market_spy_volume_zscore_63d,month,month_index
0,2006-01-31,DBC,0,-0.051731,NaN,-0.051731,-0.051314,22.0,-0.000417,next_open_to_next_open,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2006-01,0
1,2006-01-31,DIA,1,0.017902,0.016919,0.017902,0.018320,5.0,-0.000417,next_open_to_next_open,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2006-01,0
2,2006-01-31,EEM,0,-0.029831,-0.038500,-0.029831,-0.029414,21.0,-0.000417,next_open_to_next_open,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2006-01,0
3,2006-01-31,EFA,0,-0.000159,-0.007000,-0.000159,0.000258,15.0,-0.000417,next_open_to_next_open,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2006-01,0
4,2006-01-31,GLD,0,-0.013559,-0.011111,-0.013559,-0.013141,19.0,-0.000417,next_open_to_next_open,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2006-01,0
5,2006-01-31,IEF,0,-0.001123,-0.001035,-0.001123,-0.000706,17.0,-0.000417,next_open_to_next_open,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2006-01,0
6,2006-01-31,IWM,0,0.002480,0.003179,0.002480,0.002897,13.0,-0.000417,next_open_to_next_open,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2006-01,0
7,2006-01-31,IYR,1,0.022753,0.017725,0.022753,0.023171,4.0,-0.000417,next_open_to_next_open,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2006-01,0
8,2006-01-31,LQD,0,0.003611,0.006825,0.003611,0.004029,12.0,-0.000417,next_open_to_next_open,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2006-01,0
9,2006-01-31,QQQ,0,-0.012218,-0.021429,-0.012218,-0.011801,18.0,-0.000417,next_open_to_next_open,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2006-01,0


,month,rows,positive_rows,mean_forward_return
231,2025-04,25,5,0.028813
232,2025-05,25,5,0.030568
233,2025-06,25,5,0.006918
234,2025-07,25,5,0.021690
235,2025-08,25,5,0.035732
236,2025-09,25,5,0.011075
237,2025-10,25,5,0.013984
238,2025-11,25,5,0.020344
239,2025-12,25,5,0.030386
240,2026-01,25,5,0.032696


Saved tabpfn_tabicl_tactical_asset_allocation_20260511_outputs/model_frame_head.csv
Saved tabpfn_tabicl_tactical_asset_allocation_20260511_outputs/monthly_target_preview.csv


,metric,value
0,rows,6059.000000
1,months,243.000000
2,assets,25.000000
3,positive_rows,1215.000000
4,positive_rate,0.200528
5,target_top_k,5.000000


Saved tabpfn_tabicl_tactical_asset_allocation_20260511_outputs/target_definition_summary.csv


,return_mode,non_null_rows,first_signal_month,last_signal_month,is_selected_target_mode
0,close_to_close,6056,2006-01,2026-03,False
1,next_open_to_next_open,6059,2006-01,2026-03,True


Saved tabpfn_tabicl_tactical_asset_allocation_20260511_outputs/execution_return_summary.csv


PosixPath('tabpfn_tabicl_tactical_asset_allocation_20260511_outputs/execution_return_summary.csv')

## 3. Chronological Splits and Feature Policy

The model-selection period precedes the calibration and holdout periods. Feature filtering and median imputation are fitted only on the model-selection source window, then reused for calibration and holdout. This avoids using holdout distribution information when preparing model matrices.

In [4]:
date_values = pd.to_datetime(model_frame["date"])
context_mask = date_values <= pd.Timestamp(CONTEXT_END_DATE)
selection_mask = date_values <= pd.Timestamp(TUNING_END_DATE)
calibration_mask = (date_values > pd.Timestamp(TUNING_END_DATE)) & (date_values <= pd.Timestamp(CALIBRATION_END_DATE))
holdout_mask = date_values >= pd.Timestamp(HOLDOUT_START_DATE)
preholdout_mask = date_values <= pd.Timestamp(CALIBRATION_END_DATE)

context_df = model_frame.loc[context_mask].reset_index(drop=True)
selection_df = model_frame.loc[selection_mask].reset_index(drop=True)
calibration_df = model_frame.loc[calibration_mask].reset_index(drop=True)
holdout_df = model_frame.loc[holdout_mask].reset_index(drop=True)
preholdout_df = model_frame.loc[preholdout_mask].reset_index(drop=True)

if len(selection_df) == 0 or len(calibration_df) == 0 or len(holdout_df) == 0:
    raise RuntimeError("One or more required chronological windows are empty. Review split dates and data availability.")

split_summary = pd.DataFrame(
    [
        {"split": "context", "rows": len(context_df), "months": context_df["month"].nunique(), "positive_rows": int(context_df["target_top_k_next_1m"].sum()), "positive_rate": context_df["target_top_k_next_1m"].mean(), "start": context_df["date"].min(), "end": context_df["date"].max()},
        {"split": "model_selection", "rows": len(selection_df), "months": selection_df["month"].nunique(), "positive_rows": int(selection_df["target_top_k_next_1m"].sum()), "positive_rate": selection_df["target_top_k_next_1m"].mean(), "start": selection_df["date"].min(), "end": selection_df["date"].max()},
        {"split": "calibration", "rows": len(calibration_df), "months": calibration_df["month"].nunique(), "positive_rows": int(calibration_df["target_top_k_next_1m"].sum()), "positive_rate": calibration_df["target_top_k_next_1m"].mean(), "start": calibration_df["date"].min(), "end": calibration_df["date"].max()},
        {"split": "preholdout", "rows": len(preholdout_df), "months": preholdout_df["month"].nunique(), "positive_rows": int(preholdout_df["target_top_k_next_1m"].sum()), "positive_rate": preholdout_df["target_top_k_next_1m"].mean(), "start": preholdout_df["date"].min(), "end": preholdout_df["date"].max()},
        {"split": "holdout", "rows": len(holdout_df), "months": holdout_df["month"].nunique(), "positive_rows": int(holdout_df["target_top_k_next_1m"].sum()), "positive_rate": holdout_df["target_top_k_next_1m"].mean(), "start": holdout_df["date"].min(), "end": holdout_df["date"].max()},
    ]
)
display(split_summary)
save_artifact_table(split_summary, "split_summary.csv")

NON_FEATURE_COLUMNS = {
    "date",
    "month",
    "asset",
    "month_index",
    "target_top_k_next_1m",
    "forward_1m_return",
    "forward_1m_close_to_close_return",
    "forward_1m_next_open_return",
    "forward_1m_excess_return",
    "forward_1m_rank",
    "universe_forward_1m_return",
    "execution_return_mode",
}
feature_candidates = [column for column in model_frame.columns if column not in NON_FEATURE_COLUMNS]
feature_candidates = [column for column in feature_candidates if not column.startswith("target_") and not column.startswith("forward_")]
feature_candidates = [column for column in feature_candidates if pd.api.types.is_numeric_dtype(model_frame[column])]

raw_feature_candidates = list(feature_candidates)
ticker_identity_feature_columns = [column for column in raw_feature_candidates if column.startswith("asset_is_")]
group_metadata_feature_columns = [column for column in raw_feature_candidates if column.startswith("asset_group_") or column == "asset_risk_bucket"]
static_identity_feature_columns = sorted(set(ticker_identity_feature_columns + group_metadata_feature_columns))
if FEATURE_SET_VARIANT == "full":
    excluded_variant_feature_columns = []
elif FEATURE_SET_VARIANT == "ticker_ablated":
    excluded_variant_feature_columns = ticker_identity_feature_columns
elif FEATURE_SET_VARIANT == "identity_ablated":
    excluded_variant_feature_columns = static_identity_feature_columns
else:
    raise ValueError(f"Unsupported FEATURE_SET_VARIANT: {FEATURE_SET_VARIANT}")
feature_candidates = [column for column in raw_feature_candidates if column not in set(excluded_variant_feature_columns)]
feature_variant_summary = pd.DataFrame(
    [
        {"metric": "feature_set_variant", "value": FEATURE_SET_VARIANT},
        {"metric": "raw_numeric_feature_candidates", "value": len(raw_feature_candidates)},
        {"metric": "ticker_identity_feature_columns", "value": len(ticker_identity_feature_columns)},
        {"metric": "group_metadata_feature_columns", "value": len(group_metadata_feature_columns)},
        {"metric": "excluded_variant_feature_columns", "value": len(excluded_variant_feature_columns)},
        {"metric": "post_variant_feature_candidates", "value": len(feature_candidates)},
    ]
)
display(feature_variant_summary)
save_artifact_table(feature_variant_summary, "feature_variant_summary.csv", description="Feature-family exclusions used for the configured feature-set variant.")
if excluded_variant_feature_columns:
    save_artifact_table(pd.DataFrame({"excluded_feature": sorted(excluded_variant_feature_columns)}), "feature_variant_excluded_columns.csv")

selection_feature_quality = pd.DataFrame(
    {
        "feature": feature_candidates,
        "missing_rate_selection": [selection_df[column].isna().mean() for column in feature_candidates],
        "non_null_selection": [selection_df[column].notna().sum() for column in feature_candidates],
        "n_unique_selection": [selection_df[column].nunique(dropna=True) for column in feature_candidates],
    }
)
retained_feature_columns = selection_feature_quality.loc[
    (selection_feature_quality["missing_rate_selection"] <= FEATURE_MAX_MISSING_RATE)
    & (selection_feature_quality["non_null_selection"] >= FEATURE_MIN_SELECTION_OBSERVATIONS)
    & (selection_feature_quality["n_unique_selection"] > 1),
    "feature",
].tolist()

if not retained_feature_columns:
    raise RuntimeError("No retained feature columns after the feature policy. Relax feature policy thresholds or review data loading.")

missing_indicator_columns = [column for column in retained_feature_columns if selection_df[column].isna().any()]
feature_medians = selection_df[retained_feature_columns].median(numeric_only=True).replace([np.inf, -np.inf], np.nan).fillna(0.0)


def transform_features(frame):
    numeric = frame[retained_feature_columns].copy().replace([np.inf, -np.inf], np.nan)
    indicator_parts = []
    for column in missing_indicator_columns:
        indicator_parts.append(numeric[column].isna().astype(np.float32).rename(f"{column}_is_missing"))
    numeric = numeric.fillna(feature_medians).astype(np.float32)
    if indicator_parts:
        indicators = pd.concat(indicator_parts, axis=1).astype(np.float32)
        numeric = pd.concat([numeric, indicators], axis=1)
    return numeric.astype(np.float32)

X_selection = transform_features(selection_df)
y_selection = selection_df["target_top_k_next_1m"].astype(int)
X_calibration = transform_features(calibration_df)
y_calibration = calibration_df["target_top_k_next_1m"].astype(int)
X_holdout = transform_features(holdout_df)
y_holdout = holdout_df["target_top_k_next_1m"].astype(int)
X_preholdout = transform_features(preholdout_df)
y_preholdout = preholdout_df["target_top_k_next_1m"].astype(int)

feature_policy_summary = pd.DataFrame(
    [
        {"metric": "feature_set_variant", "value": FEATURE_SET_VARIANT},
        {"metric": "candidate_numeric_features_after_variant", "value": len(feature_candidates)},
        {"metric": "excluded_variant_features", "value": len(excluded_variant_feature_columns)},
        {"metric": "retained_base_features", "value": len(retained_feature_columns)},
        {"metric": "missing_indicator_features", "value": len(missing_indicator_columns)},
        {"metric": "final_model_features", "value": X_selection.shape[1]},
        {"metric": "feature_max_missing_rate", "value": FEATURE_MAX_MISSING_RATE},
        {"metric": "feature_min_selection_observations", "value": FEATURE_MIN_SELECTION_OBSERVATIONS},
    ]
)
display(feature_policy_summary)
save_artifact_table(feature_policy_summary, "feature_policy_summary.csv")
save_artifact_table(selection_feature_quality.sort_values("missing_rate_selection", ascending=False), "feature_quality_summary.csv")

feature_bundle_summary = pd.DataFrame(
    [
        {"bundle": "selection", "rows": X_selection.shape[0], "features": X_selection.shape[1], "positive_rows": int(y_selection.sum()), "positive_rate": y_selection.mean()},
        {"bundle": "calibration", "rows": X_calibration.shape[0], "features": X_calibration.shape[1], "positive_rows": int(y_calibration.sum()), "positive_rate": y_calibration.mean()},
        {"bundle": "preholdout", "rows": X_preholdout.shape[0], "features": X_preholdout.shape[1], "positive_rows": int(y_preholdout.sum()), "positive_rate": y_preholdout.mean()},
        {"bundle": "holdout", "rows": X_holdout.shape[0], "features": X_holdout.shape[1], "positive_rows": int(y_holdout.sum()), "positive_rate": y_holdout.mean()},
    ]
)
display(feature_bundle_summary)
save_artifact_table(feature_bundle_summary, "feature_bundle_summary.csv")


,split,rows,months,positive_rows,positive_rate,start,end
0,context,1784,72,360,0.201794,2006-01-31,2011-12-31
1,model_selection,3584,144,720,0.200893,2006-01-31,2017-12-31
2,calibration,600,24,120,0.200000,2018-01-31,2019-12-31
3,preholdout,4184,168,840,0.200765,2006-01-31,2019-12-31
4,holdout,1875,75,375,0.200000,2020-01-31,2026-03-31


Saved tabpfn_tabicl_tactical_asset_allocation_20260511_outputs/split_summary.csv


,metric,value
0,feature_set_variant,identity_ablated
1,raw_numeric_feature_candidates,515
2,ticker_identity_feature_columns,25
3,group_metadata_feature_columns,9
4,excluded_variant_feature_columns,34
5,post_variant_feature_candidates,481


Saved tabpfn_tabicl_tactical_asset_allocation_20260511_outputs/feature_variant_summary.csv
Saved tabpfn_tabicl_tactical_asset_allocation_20260511_outputs/feature_variant_excluded_columns.csv


,metric,value
0,feature_set_variant,identity_ablated
1,candidate_numeric_features_after_variant,481
2,excluded_variant_features,34
3,retained_base_features,481
4,missing_indicator_features,475
5,final_model_features,956
6,feature_max_missing_rate,0.35
7,feature_min_selection_observations,36


Saved tabpfn_tabicl_tactical_asset_allocation_20260511_outputs/feature_policy_summary.csv
Saved tabpfn_tabicl_tactical_asset_allocation_20260511_outputs/feature_quality_summary.csv


,bundle,rows,features,positive_rows,positive_rate
0,selection,3584,956,720,0.200893
1,calibration,600,956,120,0.200000
2,preholdout,4184,956,840,0.200765
3,holdout,1875,956,375,0.200000


Saved tabpfn_tabicl_tactical_asset_allocation_20260511_outputs/feature_bundle_summary.csv


PosixPath('tabpfn_tabicl_tactical_asset_allocation_20260511_outputs/feature_bundle_summary.csv')

## 4. Model Registry, Tuning, and Evaluation Helpers

The default learned model path is GPU-accelerated XGBoost with randomized hyperparameter search over broad distributions. Direct TabPFN and TabICL scorers are evaluated as pretrained tabular foundation model components. Deterministic rules are included because tactical allocation workflows should be compared with simple finance baselines before adding model complexity.

In [5]:
def y_to_numpy(y):
    if hasattr(y, "to_numpy"):
        return y.to_numpy(dtype=np.int32)
    return np.asarray(y, dtype=np.int32)


def to_numpy_float32(X):
    if hasattr(X, "to_numpy"):
        return X.to_numpy(dtype=np.float32)
    return np.asarray(X, dtype=np.float32)


def to_cupy_float32(X):
    return cp.asarray(to_numpy_float32(X))


def take_rows(X, row_indices):
    if hasattr(X, "iloc"):
        return X.iloc[row_indices]
    return X[row_indices]


def safe_metric(metric_fn, y_true, y_score):
    y_true_values = y_to_numpy(y_true)
    y_score_values = np.asarray(y_score, dtype=float)
    if len(np.unique(y_true_values)) < 2:
        return np.nan
    try:
        return float(metric_fn(y_true_values, y_score_values))
    except Exception:
        return np.nan


def expected_calibration_error(y_true, y_proba, n_bins=10):
    y_true = pd.Series(y_to_numpy(y_true))
    y_proba = pd.Series(np.asarray(y_proba, dtype=float)).clip(0.0, 1.0)
    if y_proba.nunique(dropna=True) <= 1:
        return np.nan
    n_bins = min(n_bins, y_proba.nunique(dropna=True))
    try:
        bins = pd.qcut(y_proba, q=n_bins, duplicates="drop")
    except ValueError:
        return np.nan
    frame = pd.DataFrame({"y_true": y_true, "y_proba": y_proba, "bin": bins})
    grouped = frame.groupby("bin", observed=True)
    weights = grouped.size() / len(frame)
    observed = grouped["y_true"].mean()
    predicted = grouped["y_proba"].mean()
    return float((weights * (observed - predicted).abs()).sum())


def calibration_bin_table(y_true, y_proba, n_bins=10):
    y_true = pd.Series(y_to_numpy(y_true))
    y_proba = pd.Series(np.asarray(y_proba, dtype=float)).clip(0.0, 1.0)
    n_bins = min(n_bins, max(1, y_proba.nunique(dropna=True)))
    try:
        bins = pd.qcut(y_proba, q=n_bins, duplicates="drop")
    except ValueError:
        bins = pd.cut(y_proba, bins=n_bins, include_lowest=True, duplicates="drop")
    frame = pd.DataFrame({"y_true": y_true, "y_proba": y_proba, "bin": bins})
    return (
        frame.groupby("bin", observed=True)
        .agg(rows=("y_true", "size"), observed_rate=("y_true", "mean"), mean_predicted_probability=("y_proba", "mean"), score_min=("y_proba", "min"), score_max=("y_proba", "max"))
        .reset_index()
    )


def positive_class_proba(model, X):
    proba = model.predict_proba(X)
    proba = cp.asnumpy(proba) if hasattr(proba, "get") else np.asarray(proba)
    if proba.ndim == 1:
        return proba.astype(float)
    if proba.shape[1] == 1:
        return proba[:, 0].astype(float)
    return proba[:, 1].astype(float)


def predict_proba_in_chunks(model, X, chunk_size=PREDICTION_CHUNK_SIZE, desc="Predicting"):
    if len(X) <= chunk_size:
        return positive_class_proba(model, X)
    parts = []
    for start in tqdm(range(0, len(X), chunk_size), desc=desc, unit="chunk", leave=False):
        stop = min(start + chunk_size, len(X))
        parts.append(positive_class_proba(model, take_rows(X, np.arange(start, stop))))
    return np.concatenate(parts)


class GPUXGBClassifier(XGBClassifier):
    def fit(self, X, y, **kwargs):
        if XGBOOST_DEVICE == "cuda":
            X_fit = to_cupy_float32(X)
            y_fit = cp.asarray(y_to_numpy(y))
        else:
            X_fit = to_numpy_float32(X)
            y_fit = y_to_numpy(y)
        return super().fit(X_fit, y_fit, **kwargs)

    def predict_proba(self, X, **kwargs):
        X_predict = to_cupy_float32(X) if XGBOOST_DEVICE == "cuda" else to_numpy_float32(X)
        proba = super().predict_proba(X_predict, **kwargs)
        return cp.asnumpy(proba) if hasattr(proba, "get") else np.asarray(proba)

    def predict(self, X, **kwargs):
        X_predict = to_cupy_float32(X) if XGBOOST_DEVICE == "cuda" else to_numpy_float32(X)
        prediction = super().predict(X_predict, **kwargs)
        return cp.asnumpy(prediction) if hasattr(prediction, "get") else np.asarray(prediction)


def make_prefit_calibrator(base_model, X_cal, y_cal, method="sigmoid"):
    try:
        calibrated = CalibratedClassifierCV(estimator=base_model, method=method, cv="prefit")
    except TypeError:
        calibrated = CalibratedClassifierCV(base_estimator=base_model, method=method, cv="prefit")
    calibrated.fit(X_cal, y_cal)
    return calibrated


def make_walk_forward_month_splits(frame, y, n_splits=CLASSICAL_TUNING_CV_SPLITS, min_train_positives=MIN_CV_TRAIN_POSITIVES, min_validation_positives=MIN_CV_VALIDATION_POSITIVES):
    y_values = y_to_numpy(y)
    months = pd.Series(frame["month"].to_numpy())
    unique_months = months.drop_duplicates().to_numpy()
    if len(unique_months) < n_splits + 2:
        n_splits = max(1, len(unique_months) - 2)
    min_train_months = max(24, int(len(unique_months) * 0.35))
    boundaries = np.unique(np.linspace(min_train_months, len(unique_months), n_splits + 1, dtype=int))
    splits = []
    rows = []
    for fold_idx, (validation_start, validation_stop) in enumerate(zip(boundaries[:-1], boundaries[1:]), start=1):
        train_months = set(unique_months[:validation_start])
        validation_months = set(unique_months[validation_start:validation_stop])
        train_idx = np.where(months.isin(train_months).to_numpy())[0]
        validation_idx = np.where(months.isin(validation_months).to_numpy())[0]
        train_pos = int(y_values[train_idx].sum()) if len(train_idx) else 0
        valid_pos = int(y_values[validation_idx].sum()) if len(validation_idx) else 0
        keep = len(train_idx) > 0 and len(validation_idx) > 0 and train_pos >= min_train_positives and valid_pos >= min_validation_positives
        rows.append(
            {
                "fold": fold_idx,
                "train_rows": len(train_idx),
                "validation_rows": len(validation_idx),
                "train_positive_rows": train_pos,
                "validation_positive_rows": valid_pos,
                "train_start": min(train_months) if train_months else "",
                "train_end": max(train_months) if train_months else "",
                "validation_start": min(validation_months) if validation_months else "",
                "validation_end": max(validation_months) if validation_months else "",
                "used": keep,
            }
        )
        if keep:
            splits.append((train_idx, validation_idx))
    if not splits:
        # Fallback to the last 20 percent of months as validation if the positive-count constraints are too strict.
        split_point = max(1, int(len(unique_months) * 0.80))
        train_months = set(unique_months[:split_point])
        validation_months = set(unique_months[split_point:])
        train_idx = np.where(months.isin(train_months).to_numpy())[0]
        validation_idx = np.where(months.isin(validation_months).to_numpy())[0]
        splits = [(train_idx, validation_idx)]
        rows.append({"fold": "fallback_last_month_block", "train_rows": len(train_idx), "validation_rows": len(validation_idx), "train_positive_rows": int(y_values[train_idx].sum()), "validation_positive_rows": int(y_values[validation_idx].sum()), "used": True})
    return splits, pd.DataFrame(rows)


class ProgressRandomizedSearchCV:
    def __init__(self, estimator, param_distributions, n_iter, cv_splits, random_state=SEED, error_score=np.nan, label="model search"):
        self.estimator = estimator
        self.param_distributions = param_distributions
        self.n_iter = int(n_iter)
        self.cv = list(cv_splits)
        self.random_state = random_state
        self.error_score = error_score
        self.label = label

    def planned_fit_count(self):
        return self.n_iter * len(self.cv)

    def fit(self, X, y):
        parameter_draws = list(ParameterSampler(self.param_distributions, n_iter=self.n_iter, random_state=self.random_state))
        if not parameter_draws:
            raise RuntimeError(f"{self.label}: no hyperparameter draws were generated.")
        if not self.cv:
            raise RuntimeError(f"{self.label}: no cross-validation folds were available.")
        trial_rows = []
        best_score = -np.inf
        best_params = None
        progress = tqdm(parameter_draws, desc=f"Tuning {self.label}", unit="trial")
        for trial_idx, params in enumerate(progress, start=1):
            fold_scores = []
            fold_errors = []
            for fold_idx, (train_idx, validation_idx) in enumerate(self.cv, start=1):
                estimator = clone(self.estimator)
                estimator.set_params(**params)
                try:
                    estimator.fit(take_rows(X, train_idx), take_rows(y, train_idx))
                    validation_score = positive_class_proba(estimator, take_rows(X, validation_idx))
                    fold_score = safe_metric(average_precision_score, take_rows(y, validation_idx), validation_score)
                except Exception as exc:
                    fold_score = self.error_score if np.isscalar(self.error_score) else np.nan
                    fold_errors.append(f"fold_{fold_idx}: {short_error(exc)}")
                fold_scores.append(fold_score)
                del estimator
                if XGBOOST_DEVICE == "cuda":
                    try:
                        cp.get_default_memory_pool().free_all_blocks()
                    except Exception:
                        pass
            fold_scores_array = np.asarray(fold_scores, dtype=float)
            mean_score = float(np.nanmean(fold_scores_array)) if np.isfinite(fold_scores_array).any() else np.nan
            std_score = float(np.nanstd(fold_scores_array)) if np.isfinite(fold_scores_array).any() else np.nan
            trial_rows.append({"trial": trial_idx, "params": params, "mean_test_score": mean_score, "std_test_score": std_score, "fold_scores": fold_scores, "errors": "; ".join(fold_errors)})
            if np.isfinite(mean_score) and mean_score > best_score:
                best_score = mean_score
                best_params = params
            progress.set_postfix(best_ap=f"{best_score:.4f}" if np.isfinite(best_score) else "nan")
        if best_params is None:
            raise RuntimeError(f"{self.label}: all randomized-search trials failed or produced NaN scores.")
        self.best_params_ = best_params
        self.best_score_ = float(best_score)
        self.cv_results_ = {
            "params": [row["params"] for row in trial_rows],
            "mean_test_score": np.asarray([row["mean_test_score"] for row in trial_rows], dtype=float),
            "std_test_score": np.asarray([row["std_test_score"] for row in trial_rows], dtype=float),
            "errors": [row["errors"] for row in trial_rows],
        }
        return self


def best_estimator_from_search(search_model):
    estimator = clone(search_model.estimator)
    estimator.set_params(**search_model.best_params_)
    return estimator


cv_splits, cv_split_summary = make_walk_forward_month_splits(selection_df, y_selection)
display(cv_split_summary)
save_artifact_table(cv_split_summary, "cv_split_summary.csv")


,fold,train_rows,validation_rows,train_positive_rows,validation_positive_rows,train_start,train_end,validation_start,validation_end,used
0,1,1234,375,250,75,2006-01,2010-02,2010-03,2011-05,True
1,2,1609,400,325,80,2006-01,2011-05,2011-06,2012-09,True
2,3,2009,400,405,80,2006-01,2012-09,2012-10,2014-01,True
3,4,2409,375,485,75,2006-01,2014-01,2014-02,2015-04,True
4,5,2784,400,560,80,2006-01,2015-04,2015-05,2016-08,True
5,6,3184,400,640,80,2006-01,2016-08,2016-09,2017-12,True


Saved tabpfn_tabicl_tactical_asset_allocation_20260511_outputs/cv_split_summary.csv


PosixPath('tabpfn_tabicl_tactical_asset_allocation_20260511_outputs/cv_split_summary.csv')

## 5. Run Allocation Scorers

This section creates deterministic rule scores, tunes the GPU XGBoost scorer, and evaluates direct TabPFN and TabICL scorers. The learned models produce probabilities for the next-month top-k label. Portfolio diagnostics use the scores only for ranking assets inside each month.

In [ ]:
def metric_row(model_name, base_model, family, calibration, evaluation_window, y_true, y_score, score_type, fit_seconds=0.0, predict_seconds=0.0, calibration_seconds=0.0, cv_average_precision=np.nan, best_params="", timing_notes=""):
    y_true_values = y_to_numpy(y_true)
    y_score_values = np.asarray(y_score, dtype=float)
    row = {
        "Model": model_name,
        "Base Model": base_model,
        "Family": family,
        "Calibration": calibration,
        "Evaluation Window": evaluation_window,
        "Rows": len(y_true_values),
        "Positive Rows": int(y_true_values.sum()),
        "Positive Rate": float(y_true_values.mean()) if len(y_true_values) else np.nan,
        "Score Type": score_type,
        "Average Precision": safe_metric(average_precision_score, y_true_values, y_score_values),
        "ROC AUC": safe_metric(roc_auc_score, y_true_values, y_score_values),
        "Brier Score": brier_score_loss(y_true_values, np.clip(y_score_values, 0.0, 1.0)) if score_type == "probability" and len(np.unique(y_true_values)) > 1 else np.nan,
        "Log Loss": log_loss(y_true_values, np.clip(y_score_values, 1e-6, 1.0 - 1e-6), labels=[0, 1]) if score_type == "probability" and len(np.unique(y_true_values)) > 1 else np.nan,
        "ECE Quantile 10": expected_calibration_error(y_true_values, y_score_values, n_bins=10) if score_type == "probability" else np.nan,
        "Fit Seconds": fit_seconds,
        "Predict Seconds": predict_seconds,
        "Calibration Seconds": calibration_seconds,
        "Workflow Seconds": fit_seconds + predict_seconds + calibration_seconds,
        "CV/Validation Average Precision": cv_average_precision,
        "Best Params": best_params,
        "Timing Notes": timing_notes,
    }
    return row


def build_prediction_frame(model_name, frame, scores, evaluation_window):
    columns = [
        "date",
        "month",
        "asset",
        "target_top_k_next_1m",
        "forward_1m_return",
        "forward_1m_close_to_close_return",
        "forward_1m_next_open_return",
        "forward_1m_excess_return",
        "forward_1m_rank",
    ]
    available_columns = [column for column in columns if column in frame.columns]
    out = frame[available_columns].copy()
    out["Model"] = model_name
    out["Evaluation Window"] = evaluation_window
    out["score"] = np.asarray(scores, dtype=np.float32)
    out["target_top_k_next_1m"] = out["target_top_k_next_1m"].astype(np.int8)
    return out


def add_model_predictions(model_name, base_model, family, calibration, score_type, eval_payloads, fit_seconds=0.0, calibration_seconds=0.0, cv_average_precision=np.nan, best_params="", timing_notes=""):
    for evaluation_window, frame, y_true, y_score, predict_seconds in eval_payloads:
        y_score_values = np.asarray(y_score, dtype=np.float32)
        predictions[(model_name, evaluation_window)] = y_score_values
        prediction_frames.append(build_prediction_frame(model_name, frame, y_score_values, evaluation_window))
        model_rows.append(metric_row(model_name, base_model, family, calibration, evaluation_window, y_true, y_score_values, score_type, fit_seconds, predict_seconds, calibration_seconds, cv_average_precision, best_params, timing_notes))


def add_rule_score(model_name, score_column, ascending=False):
    for evaluation_window, frame, y_true in [("calibration", calibration_df, y_calibration), ("holdout", holdout_df, y_holdout)]:
        if score_column not in frame.columns:
            print(f"Skipping {model_name}; missing {score_column}.")
            return
        score = frame[score_column].astype(float).to_numpy()
        if ascending:
            score = -score
        add_model_predictions(model_name, "Rule", "deterministic_rule", "none", "raw_score", [(evaluation_window, frame, y_true, score, 0.0)], timing_notes="deterministic rule score; not a calibrated probability")


# Deterministic allocation rule scores.
prior_score = np.repeat(y_preholdout.mean(), len(holdout_df))
add_model_predictions("Dummy[Preholdout prior probability]", "Dummy", "baseline", "none", "probability", [("holdout", holdout_df, y_holdout, prior_score, 0.0)], timing_notes="constant prior from pre-holdout label frequency")
add_rule_score("Rule[12M momentum top-k]", "asset_return_252d", ascending=False)
add_rule_score("Rule[6M momentum top-k]", "asset_return_126d", ascending=False)
add_rule_score("Rule[Low volatility top-k]", "asset_rv_63d", ascending=True)
add_rule_score("Rule[Risk-adjusted momentum top-k]", "asset_ma_distance_126d", ascending=False)


def xgboost_param_distributions(y):
    positive = max(int(y.sum()), 1)
    negative = max(int(len(y) - y.sum()), 1)
    scale_pos_weight = negative / positive
    return {
        "n_estimators": randint(400, 3000),
        "max_depth": randint(2, 10),
        "learning_rate": loguniform(0.003, 0.20),
        "subsample": uniform(0.55, 0.45),
        "colsample_bytree": uniform(0.45, 0.55),
        "colsample_bylevel": uniform(0.50, 0.50),
        "min_child_weight": loguniform(0.05, 80.0),
        "gamma": loguniform(1e-5, 20.0),
        "reg_alpha": loguniform(1e-5, 100.0),
        "reg_lambda": loguniform(0.05, 150.0),
        "max_bin": randint(128, 1024),
        "scale_pos_weight": [1.0, math.sqrt(scale_pos_weight), scale_pos_weight],
        "max_delta_step": [0, 1, 3, 5, 10],
    }


def make_xgboost_estimator():
    return GPUXGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method=XGBOOST_TREE_METHOD,
        device=XGBOOST_DEVICE,
        random_state=SEED,
        n_jobs=1,
        verbosity=0,
    )


def evaluate_xgboost_workflow():
    model_name = "XGBoost[GPU allocation scorer]"
    console.rule(model_name)
    search = ProgressRandomizedSearchCV(
        estimator=make_xgboost_estimator(),
        param_distributions=xgboost_param_distributions(y_selection),
        n_iter=XGBOOST_TUNING_ITERATIONS,
        cv_splits=cv_splits,
        random_state=SEED,
        label=model_name,
    )
    print(f"{model_name}: {search.n_iter} randomized parameter draws x {len(search.cv)} chronological folds = {search.planned_fit_count():,} tuning fits.")
    search_start = time.perf_counter()
    search.fit(X_selection, y_selection)
    search_seconds = time.perf_counter() - search_start
    best_params = artifact_json(search.best_params_)
    print(f"Best validation Average Precision: {search.best_score_:.4f}")
    print(f"Best params: {best_params}")

    final_model = best_estimator_from_search(search)
    final_start = time.perf_counter()
    final_model.fit(X_preholdout, y_preholdout)
    final_fit_seconds = time.perf_counter() - final_start
    predict_start = time.perf_counter()
    holdout_score = predict_proba_in_chunks(final_model, X_holdout, desc=f"{model_name} holdout prediction")
    holdout_predict_seconds = time.perf_counter() - predict_start
    add_model_predictions(model_name, "XGBoost", "gpu_boosted_tree", "none", "probability", [("holdout", holdout_df, y_holdout, holdout_score, holdout_predict_seconds)], fit_seconds=search_seconds + final_fit_seconds, cv_average_precision=search.best_score_, best_params=best_params, timing_notes="randomized search on monthly rolling-origin folds; final model fit on all pre-holdout rows")
    del final_model
    cleanup_runtime_memory("after_xgboost_final_prediction")

    if EVALUATE_CALIBRATION_VARIANTS:
        base_model = best_estimator_from_search(search)
        base_start = time.perf_counter()
        base_model.fit(X_selection, y_selection)
        base_fit_seconds = time.perf_counter() - base_start
        predict_start = time.perf_counter()
        base_calibration_score = predict_proba_in_chunks(base_model, X_calibration, desc=f"{model_name} calibration-base calibration prediction")
        base_holdout_score = predict_proba_in_chunks(base_model, X_holdout, desc=f"{model_name} calibration-base holdout prediction")
        base_predict_seconds = time.perf_counter() - predict_start
        add_model_predictions(f"{model_name} Calibration Base", "XGBoost", "gpu_boosted_tree_calibration_base", "none_calibration_base", "probability", [("calibration", calibration_df, y_calibration, base_calibration_score, base_predict_seconds / 2.0), ("holdout", holdout_df, y_holdout, base_holdout_score, base_predict_seconds / 2.0)], fit_seconds=search_seconds + base_fit_seconds, cv_average_precision=search.best_score_, best_params=best_params, timing_notes="base model excludes calibration-window labels")
        if y_calibration.nunique() > 1:
            calibration_start = time.perf_counter()
            calibrated_model = make_prefit_calibrator(base_model, X_calibration, y_calibration, method="sigmoid")
            calibration_seconds = time.perf_counter() - calibration_start
            predict_start = time.perf_counter()
            calibrated_calibration_score = predict_proba_in_chunks(calibrated_model, X_calibration, desc=f"{model_name} calibrated calibration prediction")
            calibrated_holdout_score = predict_proba_in_chunks(calibrated_model, X_holdout, desc=f"{model_name} calibrated holdout prediction")
            calibrated_predict_seconds = time.perf_counter() - predict_start
            add_model_predictions(f"{model_name} Calibrated", "XGBoost", "gpu_boosted_tree_calibrated", "sigmoid", "probability", [("calibration", calibration_df, y_calibration, calibrated_calibration_score, calibrated_predict_seconds / 2.0), ("holdout", holdout_df, y_holdout, calibrated_holdout_score, calibrated_predict_seconds / 2.0)], fit_seconds=search_seconds + base_fit_seconds, calibration_seconds=calibration_seconds, cv_average_precision=search.best_score_, best_params=best_params, timing_notes="sigmoid calibration fitted on calibration window")
            del calibrated_model, calibrated_calibration_score, calibrated_holdout_score
        del base_model, base_calibration_score, base_holdout_score
    del search
    cleanup_runtime_memory("after_xgboost_workflow")


def tabicl_batch_size():
    if CUDA_DEVICE_COUNT == 0:
        return 256
    return 1024 if len(X_preholdout) < 5000 else 512


def evaluate_direct_tfm(model_name, model_factory, package_name):
    console.rule(model_name)
    try:
        start = time.perf_counter()
        final_model = model_factory()
        print(f"{model_name}: fitting final direct model on {len(X_preholdout):,} pre-holdout rows.")
        final_model.fit(to_numpy_float32(X_preholdout), y_to_numpy(y_preholdout))
        final_fit_seconds = time.perf_counter() - start
        predict_start = time.perf_counter()
        final_holdout_score = positive_class_proba(final_model, to_numpy_float32(X_holdout))
        final_predict_seconds = time.perf_counter() - predict_start
        add_model_predictions(model_name, package_name, "direct_tfm", "none", "probability", [("holdout", holdout_df, y_holdout, final_holdout_score, final_predict_seconds)], fit_seconds=final_fit_seconds, timing_notes="direct TFM classifier fitted on all pre-holdout rows; no task-specific weight training claim is made")
        del final_model, final_holdout_score
        cleanup_runtime_memory(f"after_{model_name}_final_prediction")

        if EVALUATE_CALIBRATION_VARIANTS:
            base_model = model_factory()
            base_start = time.perf_counter()
            print(f"{model_name}: fitting calibration-base direct model on {len(X_selection):,} selection rows.")
            base_model.fit(to_numpy_float32(X_selection), y_to_numpy(y_selection))
            base_fit_seconds = time.perf_counter() - base_start
            predict_start = time.perf_counter()
            base_calibration_score = positive_class_proba(base_model, to_numpy_float32(X_calibration))
            base_holdout_score = positive_class_proba(base_model, to_numpy_float32(X_holdout))
            base_predict_seconds = time.perf_counter() - predict_start
            add_model_predictions(f"{model_name} Calibration Base", package_name, "direct_tfm_calibration_base", "none_calibration_base", "probability", [("calibration", calibration_df, y_calibration, base_calibration_score, base_predict_seconds / 2.0), ("holdout", holdout_df, y_holdout, base_holdout_score, base_predict_seconds / 2.0)], fit_seconds=base_fit_seconds, timing_notes="direct TFM context excludes calibration-window labels")
            del base_model, base_calibration_score, base_holdout_score
            cleanup_runtime_memory(f"after_{model_name}_calibration_base_prediction")
        cleanup_runtime_memory(f"after_{model_name}_workflow")
    except Exception as exc:
        error = short_error(exc)
        model_errors.append({"Model": model_name, "Error": error})
        print(f"{model_name} failed: {error}")


if RUN_GPU_XGBOOST:
    try:
        evaluate_xgboost_workflow()
    except Exception as exc:
        error = short_error(exc)
        model_errors.append({"Model": "XGBoost[GPU allocation scorer]", "Error": error})
        print(f"XGBoost workflow failed: {error}")

if RUN_LOGISTIC_REGRESSION_CPU_BENCHMARK:
    try:
        logistic_name = "LogisticRegression[CPU optional benchmark]"
        logistic = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=5000, class_weight="balanced", random_state=SEED))
        start = time.perf_counter()
        logistic.fit(X_preholdout, y_preholdout)
        fit_seconds = time.perf_counter() - start
        predict_start = time.perf_counter()
        score = positive_class_proba(logistic, X_holdout)
        predict_seconds = time.perf_counter() - predict_start
        add_model_predictions(logistic_name, "Logistic Regression", "optional_cpu", "none", "probability", [("holdout", holdout_df, y_holdout, score, predict_seconds)], fit_seconds=fit_seconds, timing_notes="optional CPU benchmark; disabled by default")
    except Exception as exc:
        model_errors.append({"Model": "LogisticRegression[CPU optional benchmark]", "Error": short_error(exc)})

if RUN_DIRECT_TABPFN:
    def make_tabpfn_direct():
        from tabpfn import TabPFNClassifier
        return TabPFNClassifier(n_estimators=N_TFM_ESTIMATORS, device=TABPFN_DEVICE, random_state=SEED)
    evaluate_direct_tfm("TabPFN[Direct allocation scorer]", make_tabpfn_direct, "TabPFN")

if RUN_DIRECT_TABICL:
    def make_tabicl_direct():
        from tabicl import TabICLClassifier
        return TabICLClassifier(n_estimators=N_TFM_ESTIMATORS, device=TABICL_DEVICE, batch_size=tabicl_batch_size(), random_state=SEED, checkpoint_version=TABICL_CHECKPOINT_VERSION)
    evaluate_direct_tfm("TabICL[Direct allocation scorer]", make_tabicl_direct, "TabICL")

allocation_summary = pd.DataFrame(model_rows)
if len(allocation_summary) == 0:
    raise RuntimeError("No model rows were produced.")
allocation_summary = allocation_summary.sort_values(["Evaluation Window", "Average Precision"], ascending=[True, False], na_position="last").reset_index(drop=True)
performance_columns = ["Model", "Base Model", "Calibration", "Evaluation Window", "Rows", "Positive Rate", "Score Type", "Average Precision", "ROC AUC", "Brier Score", "Log Loss", "ECE Quantile 10", "CV/Validation Average Precision", "Workflow Seconds"]
display(allocation_summary[performance_columns].round(4))
save_artifact_table(allocation_summary, "allocation_summary.csv")

if model_errors:
    model_error_summary = pd.DataFrame(model_errors)
    display(model_error_summary)
    save_artifact_table(model_error_summary, "model_error_summary.csv")

best_params = allocation_summary.loc[allocation_summary["Best Params"].astype(str).str.len() > 0, ["Model", "CV/Validation Average Precision", "Best Params"]].drop_duplicates(subset=["Model"])
if len(best_params) > 0:
    display(best_params)
    save_artifact_table(best_params, "best_params.csv")

timing_notes = allocation_summary.loc[allocation_summary["Timing Notes"].astype(str).str.len() > 0, ["Model", "Calibration", "Evaluation Window", "Timing Notes"]].drop_duplicates()
if len(timing_notes) > 0:
    save_artifact_table(timing_notes, "timing_notes.csv")

all_prediction_frame = pd.concat(prediction_frames, ignore_index=True) if prediction_frames else pd.DataFrame()
prediction_frames.clear()
cleanup_runtime_memory("after_prediction_frame_assembly")
if len(all_prediction_frame) > 0 and SAVE_FULL_PREDICTION_SCORES:
    save_artifact_table(all_prediction_frame, "model_prediction_scores.csv", description="Per-row model scores for calibration and holdout windows.")
elif len(all_prediction_frame) > 0:
    compact_prediction_scores = all_prediction_frame.loc[
        all_prediction_frame["Evaluation Window"].eq("holdout"),
        ["month", "asset", "target_top_k_next_1m", "forward_1m_return", "Model", "score"],
    ].copy()
    save_artifact_table(compact_prediction_scores, "model_prediction_scores_holdout_compact.csv", description="Compact holdout-only model scores for memory-efficient runs.")


───────────────────────────────────────── XGBoost[GPU allocation scorer] ──────────────────────────────────────────

XGBoost[GPU allocation scorer]: 160 randomized parameter draws x 6 chronological folds = 960 tuning fits.


Tuning XGBoost[GPU allocation scorer]:   0%|          | 0/160 [00:00<?, ?trial/s]

Best validation Average Precision: 0.2646
Best params: {"colsample_bylevel": 0.5054188257401492, "colsample_bytree": 0.9479600870305951, "gamma": 3.760106004074087e-05, "learning_rate": 0.011468858364579492, "max_bin": 845, "max_delta_step": 5, "max_depth": 4, "min_child_weight": 1.899692754725157, "n_estimators": 543, "reg_alpha": 0.013775857884996356, "reg_lambda": 0.5229972565439126, "scale_pos_weight": 1.0, "subsample": 0.7991663735556104}


/usr/local/lib/python3.12/dist-packages/sklearn/calibration.py:333: UserWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


...

TabPFN[Direct allocation scorer]: fitting calibration-base direct model on 3,584 selection rows.


──────────────────────────────────────── TabICL[Direct allocation scorer] ─────────────────────────────────────────

TabICL[Direct allocation scorer]: fitting final direct model on 4,184 pre-holdout rows.
INFO: You are downloading 'tabicl-classifier-v2-20260212.ckpt', the latest best-performing version, used in our TabICLv2 paper.

Checkpoint 'tabicl-classifier-v2-20260212.ckpt' not cached.



tabicl-classifier-v2-20260212.ckpt:   0%|          | 0.00/110M [00:00<?, ?B/s]

TabICL[Direct allocation scorer]: fitting calibration-base direct model on 3,584 selection rows.


## 6. Ranking, Portfolio, Calibration, Runtime, and Uncertainty Diagnostics

A tactical allocation score should be evaluated in several views: row-level classification quality, within-month rank quality, portfolio behavior after transaction costs, runtime, and probability diagnostics. Constant-score baselines are useful for classification calibration checks but do not define a cross-sectional top-k allocation without an arbitrary tie-break rule, so they are excluded from score-driven portfolio diagnostics. Deterministic rule portfolios that are already represented as named benchmark portfolios are not duplicated in the portfolio table. The portfolio backtest here is a diagnostic of score utility, not a trading recommendation.

In [ ]:
MODEL_DISPLAY_NAMES = {
    "Rule[12M momentum top-k]": "12M momentum",
    "Rule[6M momentum top-k]": "6M momentum",
    "Rule[Low volatility top-k]": "Low volatility",
    "Rule[Risk-adjusted momentum top-k]": "Risk-adjusted momentum",
    "XGBoost[GPU allocation scorer]": "XGBoost final",
    "XGBoost[GPU allocation scorer] Calibration Base": "XGBoost calibration base",
    "XGBoost[GPU allocation scorer] Calibrated": "XGBoost calibrated",
    "TabPFN[Direct allocation scorer]": "TabPFN direct",
    "TabPFN[Direct allocation scorer] Calibration Base": "TabPFN calibration base",
    "TabICL[Direct allocation scorer]": "TabICL direct",
    "TabICL[Direct allocation scorer] Calibration Base": "TabICL calibration base",
    "Dummy[Preholdout prior probability]": "Preholdout prior",
    "Equal weight universe": "Equal weight",
    "SPY only": "SPY only",
    "60/40 SPY/TLT": "60/40 SPY/TLT",
    "12M momentum top-k portfolio": "12M momentum",
    "Low volatility top-k portfolio": "Low volatility",
}


DUPLICATE_DETERMINISTIC_PORTFOLIO_MODELS = {
    "Rule[12M momentum top-k]": "12M momentum top-k portfolio",
    "Rule[Low volatility top-k]": "Low volatility top-k portfolio",
}


def model_display_name(model_name):
    return MODEL_DISPLAY_NAMES.get(model_name, model_name)


def month_index_to_timestamp(index):
    values = pd.Index(index).astype(str)
    try:
        return pd.PeriodIndex(values, freq="M").to_timestamp()
    except Exception:
        return pd.to_datetime(values, errors="coerce")


def selected_publication_rows(summary, evaluation_window="holdout"):
    selected = summary.loc[(summary["Evaluation Window"] == evaluation_window) & (summary["Model"].isin(PUBLICATION_MODEL_ORDER))].copy()
    if len(selected) == 0:
        selected = summary.loc[(summary["Evaluation Window"] == evaluation_window) & summary["Average Precision"].notna()].copy()
    order_map = {name: idx for idx, name in enumerate(PUBLICATION_MODEL_ORDER)}
    selected["publication_order"] = selected["Model"].map(order_map).fillna(999).astype(int)
    return selected.sort_values(["publication_order", "Average Precision"], ascending=[True, False]).drop(columns=["publication_order"])


def monthly_rank_quality(score_frame, top_k=TARGET_TOP_K, month_column="month", score_column="score"):
    rows = []
    required_columns = ["Model", "Evaluation Window", month_column, score_column, "forward_1m_return", "target_top_k_next_1m"]
    optional_columns = ["asset"] if "asset" in score_frame.columns else []
    working = score_frame[required_columns + optional_columns].copy()
    working[score_column] = pd.to_numeric(working[score_column], errors="coerce")
    working["forward_1m_return"] = pd.to_numeric(working["forward_1m_return"], errors="coerce")
    working = working.dropna(subset=[score_column, "forward_1m_return"])
    group_columns = ["Model", "Evaluation Window", month_column]
    for (model_name, evaluation_window, month), numeric_group in working.groupby(group_columns, sort=False):
        if len(numeric_group) < max(2, top_k):
            continue

        score = numeric_group[score_column].to_numpy(dtype=np.float64, copy=False)
        realized = numeric_group["forward_1m_return"].to_numpy(dtype=np.float64, copy=False)
        unique_score_count = np.unique(score[np.isfinite(score)]).size
        universe_mean = float(np.nanmean(realized))

        if unique_score_count <= 1:
            ic = np.nan
            top_k_hit_rate = np.nan
            selected_mean_return = np.nan
            active_return = np.nan
        else:
            if "asset" in numeric_group.columns:
                sort_values = numeric_group.sort_values([score_column, "asset"], ascending=[False, True]).head(top_k)
                selected_returns = sort_values["forward_1m_return"].to_numpy(dtype=np.float64, copy=False)
                selected_targets = sort_values["target_top_k_next_1m"].to_numpy(dtype=np.float64, copy=False)
            else:
                top_indices = np.argsort(-score, kind="mergesort")[:top_k]
                selected_returns = realized[top_indices]
                selected_targets = numeric_group["target_top_k_next_1m"].to_numpy(dtype=np.float64, copy=False)[top_indices]
            ic = spearmanr(score, realized, nan_policy="omit").correlation if np.unique(realized[np.isfinite(realized)]).size > 1 else np.nan
            top_k_hit_rate = float(np.nanmean(selected_targets))
            selected_mean_return = float(np.nanmean(selected_returns))
            active_return = selected_mean_return - universe_mean

        rows.append(
            {
                "Model": model_name,
                "Evaluation Window": evaluation_window,
                "month": month,
                "rows": len(numeric_group),
                "top_k": top_k,
                "spearman_ic": float(ic) if ic is not None else np.nan,
                "top_k_hit_rate": top_k_hit_rate,
                "selected_mean_forward_return": selected_mean_return,
                "universe_mean_forward_return": universe_mean,
                "active_forward_return": active_return,
            }
        )
    return pd.DataFrame(rows)


monthly_rank_summary = monthly_rank_quality(all_prediction_frame, top_k=TARGET_TOP_K) if len(all_prediction_frame) else pd.DataFrame()
if len(monthly_rank_summary) > 0:
    monthly_rank_aggregate = (
        monthly_rank_summary.groupby(["Model", "Evaluation Window"])
        .agg(
            months=("month", "nunique"),
            mean_spearman_ic=("spearman_ic", "mean"),
            median_spearman_ic=("spearman_ic", "median"),
            mean_top_k_hit_rate=("top_k_hit_rate", "mean"),
            mean_selected_forward_return=("selected_mean_forward_return", "mean"),
            mean_universe_forward_return=("universe_mean_forward_return", "mean"),
            mean_active_forward_return=("active_forward_return", "mean"),
        )
        .reset_index()
        .sort_values(["Evaluation Window", "mean_active_forward_return"], ascending=[True, False])
    )
    display(monthly_rank_aggregate.round(4))
    save_artifact_table(monthly_rank_summary, "monthly_rank_summary.csv")
    save_artifact_table(monthly_rank_aggregate, "monthly_rank_aggregate.csv")


def has_monthly_rank_signal(score_frame, score_column="score"):
    unique_counts = score_frame.groupby("month")[score_column].nunique(dropna=True)
    return bool(len(unique_counts) > 0 and (unique_counts > 1).all())


def weights_from_score_frame(score_frame, score_column="score", top_k=TARGET_TOP_K):
    months = sorted(score_frame["month"].unique())
    assets = sorted(score_frame["asset"].unique())
    weights = pd.DataFrame(0.0, index=months, columns=assets)
    for month, group in score_frame.groupby("month"):
        sort_columns = [score_column]
        ascending = [False]
        if "asset" in group.columns:
            sort_columns.append("asset")
            ascending.append(True)
        selected = group.sort_values(sort_columns, ascending=ascending).head(min(top_k, len(group)))
        if len(selected) == 0:
            continue
        weights.loc[month, selected["asset"].tolist()] = 1.0 / len(selected)
    return weights


def apply_turnover_cap(weights, turnover_cap):
    if turnover_cap is None or not np.isfinite(turnover_cap):
        return weights.copy()
    capped = weights.copy().fillna(0.0).astype(float)
    if len(capped) == 0:
        return capped
    previous = pd.Series(0.0, index=capped.columns)
    adjusted_rows = []
    for _, target in capped.iterrows():
        target = target.astype(float).fillna(0.0)
        if previous.abs().sum() == 0.0:
            adjusted = target
        else:
            diff = target - previous
            turnover = float(diff.abs().sum())
            if turnover > turnover_cap and turnover > 0.0:
                adjusted = previous + diff * (turnover_cap / turnover)
            else:
                adjusted = target
        adjusted = adjusted.clip(lower=0.0)
        total_weight = float(adjusted.sum())
        if total_weight > 0.0:
            adjusted = adjusted / total_weight
        adjusted_rows.append(adjusted)
        previous = adjusted
    return pd.DataFrame(adjusted_rows, index=capped.index, columns=capped.columns)


def deterministic_weight_frames():
    holdout_base = holdout_df.copy()
    months = sorted(holdout_base["month"].unique())
    assets = sorted(holdout_base["asset"].unique())
    frames = {}

    equal_weight = pd.DataFrame(0.0, index=months, columns=assets)
    for month, group in holdout_base.groupby("month"):
        available = group["asset"].tolist()
        equal_weight.loc[month, available] = 1.0 / len(available)
    frames["Equal weight universe"] = equal_weight

    spy_only = pd.DataFrame(0.0, index=months, columns=assets)
    if "SPY" in assets:
        spy_only["SPY"] = 1.0
        frames["SPY only"] = spy_only

    sixty_forty = pd.DataFrame(0.0, index=months, columns=assets)
    if "SPY" in assets and "TLT" in assets:
        sixty_forty["SPY"] = 0.60
        sixty_forty["TLT"] = 0.40
        frames["60/40 SPY/TLT"] = sixty_forty

    inv_vol = pd.DataFrame(0.0, index=months, columns=assets)
    for month, group in holdout_base.groupby("month"):
        vol = group.set_index("asset")["asset_rv_63d"].astype(float).replace(0.0, np.nan)
        raw = 1.0 / vol.clip(lower=0.02)
        raw = raw.replace([np.inf, -np.inf], np.nan).dropna()
        if len(raw) > 0:
            inv_vol.loc[month, raw.index] = raw / raw.sum()
    frames["Inverse volatility universe"] = inv_vol

    momentum_scores = holdout_base[["month", "asset", "asset_return_252d"]].rename(columns={"asset_return_252d": "score"}).copy()
    frames["12M momentum top-k portfolio"] = weights_from_score_frame(momentum_scores, top_k=TARGET_TOP_K)
    low_vol_scores = holdout_base[["month", "asset", "asset_rv_63d"]].rename(columns={"asset_rv_63d": "score"}).copy()
    low_vol_scores["score"] = -low_vol_scores["score"].astype(float)
    frames["Low volatility top-k portfolio"] = weights_from_score_frame(low_vol_scores, top_k=TARGET_TOP_K)
    return frames


def returns_pivot_for_frame(frame):
    return frame.pivot_table(index="month", columns="asset", values="forward_1m_return", aggfunc="first").sort_index()


def portfolio_metrics_from_weights(weights, returns, transaction_cost_bps=TRANSACTION_COST_BPS):
    weights = weights.reindex(index=returns.index, columns=returns.columns).fillna(0.0)
    returns = returns.reindex_like(weights).fillna(0.0)
    gross_returns = (weights * returns).sum(axis=1)
    turnover = weights.diff().abs().sum(axis=1)
    if len(turnover) > 0:
        turnover.iloc[0] = weights.iloc[0].abs().sum()
    costs = turnover * transaction_cost_bps / 10000.0
    net_returns = gross_returns - costs
    equity = (1.0 + net_returns).cumprod()
    years = len(net_returns) / MONTHS_PER_YEAR
    cagr = equity.iloc[-1] ** (1.0 / years) - 1.0 if years > 0 and equity.iloc[-1] > 0 else np.nan
    ann_vol = net_returns.std(ddof=1) * math.sqrt(MONTHS_PER_YEAR)
    sharpe = net_returns.mean() / net_returns.std(ddof=1) * math.sqrt(MONTHS_PER_YEAR) if net_returns.std(ddof=1) > 0 else np.nan
    downside = net_returns[net_returns < 0.0]
    sortino = net_returns.mean() / downside.std(ddof=1) * math.sqrt(MONTHS_PER_YEAR) if len(downside) > 1 and downside.std(ddof=1) > 0 else np.nan
    drawdown = equity / equity.cummax() - 1.0
    return {
        "Months": len(net_returns),
        "CAGR": cagr,
        "Annualized Volatility": ann_vol,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "Max Drawdown": drawdown.min(),
        "Average Monthly Return": net_returns.mean(),
        "Annualized Turnover": turnover.sum() / years if years > 0 else np.nan,
        "Average Holdings": (weights > 0).sum(axis=1).mean(),
        "Total Transaction Cost": costs.sum(),
        "Final Equity": equity.iloc[-1],
    }, equity, net_returns, turnover


holdout_returns = returns_pivot_for_frame(holdout_df)
portfolio_rows = []
equity_curves = pd.DataFrame(index=holdout_returns.index)
monthly_strategy_returns = pd.DataFrame(index=holdout_returns.index)
turnover_curves = pd.DataFrame(index=holdout_returns.index)
weight_frames = {}

for name, weights in deterministic_weight_frames().items():
    metrics, equity, net_returns, turnover = portfolio_metrics_from_weights(weights, holdout_returns)
    metrics.update({"Strategy": name, "Source": "deterministic"})
    portfolio_rows.append(metrics)
    equity_curves[name] = equity
    monthly_strategy_returns[name] = net_returns
    turnover_curves[name] = turnover
    weight_frames[name] = weights

selected_holdout_predictions = all_prediction_frame.loc[all_prediction_frame["Evaluation Window"] == "holdout"].copy()
portfolio_exclusion_rows = []
score_portfolio_groups = []
for model_name, score_frame in selected_holdout_predictions.groupby("Model"):
    duplicate_strategy = DUPLICATE_DETERMINISTIC_PORTFOLIO_MODELS.get(model_name)
    if duplicate_strategy in weight_frames:
        portfolio_exclusion_rows.append(
            {
                "Model": model_name,
                "Reason": f"excluded from score-driven portfolio because it duplicates deterministic benchmark {duplicate_strategy}",
                "Holdout Months": score_frame["month"].nunique(),
            }
        )
        continue
    if not has_monthly_rank_signal(score_frame):
        portfolio_exclusion_rows.append(
            {
                "Model": model_name,
                "Reason": "excluded from score-driven portfolio because at least one holdout month has no cross-sectional score variation",
                "Holdout Months": score_frame["month"].nunique(),
            }
        )
        continue
    score_portfolio_groups.append((model_name, score_frame))

if portfolio_exclusion_rows:
    portfolio_exclusion_summary = pd.DataFrame(portfolio_exclusion_rows)
    display(portfolio_exclusion_summary)
    save_artifact_table(portfolio_exclusion_summary, "portfolio_exclusion_summary.csv")
else:
    portfolio_exclusion_summary = pd.DataFrame(columns=["Model", "Reason", "Holdout Months"])

for model_name, score_frame in tqdm(score_portfolio_groups, desc="Building score-driven portfolios", unit="model"):
    weights = weights_from_score_frame(score_frame, top_k=TARGET_TOP_K)
    metrics, equity, net_returns, turnover = portfolio_metrics_from_weights(weights, holdout_returns)
    metrics.update({"Strategy": model_name, "Source": "model_score"})
    portfolio_rows.append(metrics)
    equity_curves[model_name] = equity
    monthly_strategy_returns[model_name] = net_returns
    turnover_curves[model_name] = turnover
    weight_frames[model_name] = weights

portfolio_summary = pd.DataFrame(portfolio_rows).sort_values(["Sharpe", "Final Equity"], ascending=[False, False]).reset_index(drop=True)
display(portfolio_summary.round(4))
save_artifact_table(portfolio_summary, "portfolio_summary.csv", description="Holdout monthly allocation diagnostics after transaction costs.")
save_artifact_table(equity_curves.reset_index(names="month"), "portfolio_equity_curves.csv")
save_artifact_table(monthly_strategy_returns.reset_index(names="month"), "portfolio_monthly_returns.csv")
save_artifact_table(turnover_curves.reset_index(names="month"), "portfolio_turnover.csv")

# Store weights in long format for offline inspection.
weight_rows = []
for strategy, weights in weight_frames.items():
    long_weights = weights.reset_index(names="month").melt(id_vars="month", var_name="asset", value_name="weight")
    long_weights["Strategy"] = strategy
    weight_rows.append(long_weights)
portfolio_weights_long = pd.concat(weight_rows, ignore_index=True) if weight_rows else pd.DataFrame()
if len(portfolio_weights_long) > 0:
    save_artifact_table(portfolio_weights_long, "portfolio_weights_long.csv")

portfolio_cost_sensitivity_rows = []
for cost_bps in TRANSACTION_COST_SENSITIVITY_BPS:
    for strategy, weights in weight_frames.items():
        metrics, _, _, _ = portfolio_metrics_from_weights(weights, holdout_returns, transaction_cost_bps=cost_bps)
        metrics.update({"Strategy": strategy, "Transaction Cost Bps": cost_bps})
        portfolio_cost_sensitivity_rows.append(metrics)
portfolio_transaction_cost_sensitivity = pd.DataFrame(portfolio_cost_sensitivity_rows)
if len(portfolio_transaction_cost_sensitivity) > 0:
    portfolio_transaction_cost_sensitivity = portfolio_transaction_cost_sensitivity.sort_values(["Transaction Cost Bps", "Sharpe", "Final Equity"], ascending=[True, False, False]).reset_index(drop=True)
    display(portfolio_transaction_cost_sensitivity.round(4))
    save_artifact_table(portfolio_transaction_cost_sensitivity, "portfolio_transaction_cost_sensitivity.csv", description="Holdout portfolio diagnostics under alternative one-way transaction cost assumptions.")

portfolio_turnover_constraint_rows = []
portfolio_turnover_constraint_monthly_returns = pd.DataFrame(index=holdout_returns.index)
for turnover_cap in TURNOVER_CONSTRAINT_CAPS:
    for strategy, weights in weight_frames.items():
        capped_weights = apply_turnover_cap(weights, turnover_cap)
        metrics, _, net_returns, turnover = portfolio_metrics_from_weights(capped_weights, holdout_returns)
        metrics.update({"Strategy": strategy, "Turnover Cap": turnover_cap, "Constraint Source": "post_score_turnover_cap"})
        portfolio_turnover_constraint_rows.append(metrics)
        portfolio_turnover_constraint_monthly_returns[f"{strategy} | turnover_cap_{turnover_cap:g}"] = net_returns
portfolio_turnover_constraint_summary = pd.DataFrame(portfolio_turnover_constraint_rows)
if len(portfolio_turnover_constraint_summary) > 0:
    portfolio_turnover_constraint_summary = portfolio_turnover_constraint_summary.sort_values(["Turnover Cap", "Sharpe", "Final Equity"], ascending=[True, False, False]).reset_index(drop=True)
    display(portfolio_turnover_constraint_summary.round(4))
    save_artifact_table(portfolio_turnover_constraint_summary, "portfolio_turnover_constraint_summary.csv", description="Holdout portfolio diagnostics after applying monthly turnover caps to target weights.")
    save_artifact_table(portfolio_turnover_constraint_monthly_returns.reset_index(names="month"), "portfolio_turnover_constraint_monthly_returns.csv")

# Precision-recall curves.
publication_holdout = selected_publication_rows(allocation_summary, "holdout")
if len(publication_holdout) > 0:
    fig, ax = plt.subplots(figsize=(8, 5.5))
    y_true = y_to_numpy(y_holdout)
    base_rate = y_true.mean()
    for _, row in publication_holdout.iterrows():
        model_name = row["Model"]
        score = predictions.get((model_name, "holdout"))
        if score is None:
            continue
        precision, recall, _ = precision_recall_curve(y_true, np.asarray(score, dtype=float))
        ax.plot(recall, precision, linewidth=1.8, label=f"{model_display_name(model_name)} (AP {row['Average Precision']:.3f})")
    ax.axhline(base_rate, color="gray", linestyle="--", linewidth=1, label=f"Base rate {base_rate:.3f}")
    ax.set_title("Holdout Precision-Recall Curves")
    ax.set_xlabel("Recall")
    ax.set_ylabel("Precision")
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8, frameon=False)
    ax.grid(alpha=0.25)
    fig.subplots_adjust(right=0.62)
    output_path = ARTIFACT_DIR / "publication_precision_recall_curves_holdout.png"
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    register_artifact(output_path, "figure", "Holdout precision-recall curves for selected models.")
    print(f"Saved {output_path}")

# Runtime versus AP.
runtime_ap_summary = allocation_summary.loc[
    allocation_summary["Average Precision"].notna(),
    ["Model", "Base Model", "Calibration", "Evaluation Window", "Average Precision", "ROC AUC", "Brier Score", "Workflow Seconds"],
].sort_values(["Evaluation Window", "Average Precision"], ascending=[True, False]).reset_index(drop=True)
display(runtime_ap_summary.round(4))
save_artifact_table(runtime_ap_summary, "runtime_ap_summary.csv")

holdout_runtime = selected_publication_rows(allocation_summary, "holdout")
holdout_runtime = holdout_runtime.loc[holdout_runtime["Average Precision"].notna()].copy()
if len(holdout_runtime) > 0:
    fig, ax = plt.subplots(figsize=(8.5, 5.5))
    for _, row in holdout_runtime.iterrows():
        label = f"{model_display_name(row['Model'])} ({row['Workflow Seconds']:.1f}s, AP {row['Average Precision']:.3f})"
        ax.scatter(row["Workflow Seconds"], row["Average Precision"], s=60, label=label)
    ax.set_xscale("symlog", linthresh=1.0)
    ax.set_title("Workflow Time versus Holdout Average Precision")
    ax.set_xlabel("Workflow seconds, symlog scale")
    ax.set_ylabel("Average Precision")
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8, frameon=False)
    ax.grid(alpha=0.25)
    fig.subplots_adjust(right=0.62)
    output_path = ARTIFACT_DIR / "publication_runtime_vs_average_precision_holdout.png"
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    register_artifact(output_path, "figure", "Runtime versus holdout Average Precision for publication models.")
    print(f"Saved {output_path}")

# Portfolio equity curves.
if len(equity_curves.columns) > 0:
    selected_equity_candidates = [
        "Equal weight universe",
        "SPY only",
        "60/40 SPY/TLT",
        "12M momentum top-k portfolio",
        "XGBoost[GPU allocation scorer]",
        "XGBoost[GPU allocation scorer] Calibrated",
        "TabPFN[Direct allocation scorer]",
        "TabICL[Direct allocation scorer]",
    ]
    selected_equity_columns = [column for column in selected_equity_candidates if column in equity_curves.columns]
    if not selected_equity_columns:
        selected_equity_columns = list(equity_curves.columns[:8])
    fig, ax = plt.subplots(figsize=(9.5, 5.5))
    plot_index = month_index_to_timestamp(equity_curves.index)
    for column in selected_equity_columns:
        ax.plot(plot_index, equity_curves[column], linewidth=1.8, label=model_display_name(column))
    ax.set_title("Holdout Tactical Allocation Diagnostic")
    ax.set_xlabel("Year")
    ax.set_ylabel("Growth of 1.0 before taxes")
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
    ax.xaxis.set_minor_locator(mdates.MonthLocator(interval=3))
    ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8, frameon=False)
    ax.grid(alpha=0.25)
    fig.subplots_adjust(right=0.60, bottom=0.16)
    output_path = ARTIFACT_DIR / "portfolio_equity_curves.png"
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    register_artifact(output_path, "figure", "Holdout portfolio equity curves for selected strategies.")
    print(f"Saved {output_path}")

# Calibration diagnostics.
calibration_quality_view = allocation_summary.loc[
    (allocation_summary["Score Type"] == "probability") & allocation_summary["Average Precision"].notna(),
    ["Model", "Family", "Calibration", "Evaluation Window", "Average Precision", "Brier Score", "Log Loss", "ECE Quantile 10"],
].sort_values(["Evaluation Window", "Average Precision"], ascending=[True, False]).reset_index(drop=True)
display(calibration_quality_view.round(4))
save_artifact_table(calibration_quality_view, "calibration_quality_view.csv")

reliability_rows = []
for (model_name, evaluation_window), score in predictions.items():
    if evaluation_window not in {"calibration", "holdout"}:
        continue
    frame = calibration_df if evaluation_window == "calibration" else holdout_df
    y_true = y_calibration if evaluation_window == "calibration" else y_holdout
    if model_name not in allocation_summary.loc[allocation_summary["Score Type"] == "probability", "Model"].values:
        continue
    table = calibration_bin_table(y_true, score, n_bins=10)
    table.insert(0, "Evaluation Window", evaluation_window)
    table.insert(0, "Model", model_name)
    reliability_rows.append(table)
reliability_summary = pd.concat(reliability_rows, ignore_index=True) if reliability_rows else pd.DataFrame()
if len(reliability_summary) > 0:
    save_artifact_table(reliability_summary, "reliability_summary.csv")

calibration_models = [name for name in CALIBRATION_MODEL_ORDER if (name, "holdout") in predictions]
if calibration_models:
    calibration_points = []
    for model_name in calibration_models:
        score = predictions.get((model_name, "holdout"))
        if score is None:
            continue
        prob_true, prob_pred = calibration_curve(y_to_numpy(y_holdout), np.clip(score, 0.0, 1.0), n_bins=10, strategy="quantile")
        calibration_points.append((model_name, prob_pred, prob_true))
    if calibration_points:
        all_values = np.concatenate([np.asarray(values, dtype=float) for _, prob_pred, prob_true in calibration_points for values in [prob_pred, prob_true]])
        axis_min = max(0.0, float(np.nanmin(all_values)) - 0.05)
        axis_max = min(1.0, float(np.nanmax(all_values)) + 0.05)
        if axis_max - axis_min < 0.25:
            midpoint = (axis_min + axis_max) / 2.0
            axis_min = max(0.0, midpoint - 0.125)
            axis_max = min(1.0, midpoint + 0.125)
        fig, ax = plt.subplots(figsize=(7.5, 5.5))
        for model_name, prob_pred, prob_true in calibration_points:
            ax.plot(prob_pred, prob_true, marker="o", linewidth=1.8, label=model_display_name(model_name))
        ax.plot([axis_min, axis_max], [axis_min, axis_max], color="gray", linestyle="--", linewidth=1, label="Perfect calibration")
        ax.set_xlim(axis_min, axis_max)
        ax.set_ylim(axis_min, axis_max)
        ax.set_title("Holdout Calibration Diagnostics")
        ax.set_xlabel("Mean predicted probability")
        ax.set_ylabel("Observed top-k frequency")
        ax.legend(loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=8, frameon=False)
        ax.grid(alpha=0.25)
        fig.subplots_adjust(right=0.62)
        output_path = ARTIFACT_DIR / "publication_calibration_curves_holdout.png"
        fig.savefig(output_path, dpi=180, bbox_inches="tight")
        plt.close(fig)
        register_artifact(output_path, "figure", "Holdout calibration curves for selected probability models.")
        print(f"Saved {output_path}")

# Month-block bootstrap uncertainty for row metrics and portfolio returns.
def bootstrap_quantiles(values, confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    alpha = 1.0 - confidence_level
    lower, median, upper = np.quantile(values, [alpha / 2.0, 0.5, 1.0 - alpha / 2.0])
    return lower, median, upper


def portfolio_return_point_metrics(values):
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return {"mean_monthly_return": np.nan, "cagr": np.nan, "annualized_volatility": np.nan, "sharpe": np.nan}
    years = len(values) / MONTHS_PER_YEAR
    ending_value = float(np.prod(1.0 + values))
    cagr = ending_value ** (1.0 / years) - 1.0 if years > 0 and ending_value > 0 else np.nan
    volatility = float(np.std(values, ddof=1) * math.sqrt(MONTHS_PER_YEAR)) if len(values) > 1 else np.nan
    sharpe = float(np.mean(values) / np.std(values, ddof=1) * math.sqrt(MONTHS_PER_YEAR)) if len(values) > 1 and np.std(values, ddof=1) > 0 else np.nan
    return {
        "mean_monthly_return": float(np.mean(values)),
        "cagr": cagr,
        "annualized_volatility": volatility,
        "sharpe": sharpe,
    }


def bootstrap_portfolio_return_metrics(monthly_returns, n_iterations=BOOTSTRAP_ITERATIONS):
    if n_iterations <= 0 or len(monthly_returns) == 0:
        return pd.DataFrame()
    rng = np.random.default_rng(SEED + 17)
    rows = []
    for strategy in tqdm(list(monthly_returns.columns), desc="Bootstrap portfolio returns", unit="strategy"):
        values = pd.to_numeric(monthly_returns[strategy], errors="coerce").dropna().to_numpy(dtype=float)
        if len(values) < 3:
            continue
        point_metrics = portfolio_return_point_metrics(values)
        boot_metrics = {name: [] for name in point_metrics}
        for _ in range(n_iterations):
            sampled = rng.choice(values, size=len(values), replace=True)
            sampled_metrics = portfolio_return_point_metrics(sampled)
            for metric_name, metric_value in sampled_metrics.items():
                boot_metrics[metric_name].append(metric_value)
        for metric_name, metric_values in boot_metrics.items():
            lower, median, upper = bootstrap_quantiles(metric_values)
            rows.append(
                {
                    "Strategy": strategy,
                    "Metric": metric_name,
                    "Point Estimate": point_metrics.get(metric_name, np.nan),
                    "Bootstrap Median": median,
                    "CI Lower": lower,
                    "CI Upper": upper,
                    "Bootstrap Iterations Requested": n_iterations,
                    "Bootstrap Iterations Used": len([value for value in metric_values if np.isfinite(value)]),
                    "Confidence Level": BOOTSTRAP_CONFIDENCE_LEVEL,
                }
            )
    return pd.DataFrame(rows)


publication_portfolio_bootstrap_uncertainty = bootstrap_portfolio_return_metrics(monthly_strategy_returns, n_iterations=BOOTSTRAP_ITERATIONS)
if len(publication_portfolio_bootstrap_uncertainty) > 0:
    display(publication_portfolio_bootstrap_uncertainty.round(4))
    save_artifact_table(publication_portfolio_bootstrap_uncertainty, "publication_portfolio_month_bootstrap_uncertainty.csv")
cleanup_runtime_memory("after_portfolio_bootstrap")


def _rankdata_average(values):
    values = np.asarray(values, dtype=np.float64)
    order = np.argsort(values, kind="mergesort")
    ranks = np.empty(len(values), dtype=np.float64)
    sorted_values = values[order]
    start = 0
    while start < len(values):
        stop = start + 1
        while stop < len(values) and sorted_values[stop] == sorted_values[start]:
            stop += 1
        rank = 0.5 * (start + 1 + stop)
        ranks[order[start:stop]] = rank
        start = stop
    return ranks


def _spearman_corr_fast(a, b):
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() < 2:
        return np.nan
    a = a[mask]
    b = b[mask]
    if np.unique(a).size <= 1 or np.unique(b).size <= 1:
        return np.nan
    ra = _rankdata_average(a)
    rb = _rankdata_average(b)
    ra = ra - ra.mean()
    rb = rb - rb.mean()
    denom = np.sqrt(np.sum(ra * ra) * np.sum(rb * rb))
    return float(np.sum(ra * rb) / denom) if denom > 0 else np.nan


def _monthly_rank_arrays(y_values, score_values, return_values, month_codes, top_k=TARGET_TOP_K):
    ic_values = []
    hit_values = []
    for month_code in np.unique(month_codes):
        idx = np.flatnonzero(month_codes == month_code)
        if len(idx) < max(2, top_k):
            continue
        scores = score_values[idx]
        realized = return_values[idx]
        targets = y_values[idx]
        valid = np.isfinite(scores) & np.isfinite(realized)
        if valid.sum() < max(2, top_k):
            continue
        scores = scores[valid]
        realized = realized[valid]
        targets = targets[valid]
        if np.unique(scores).size <= 1:
            continue
        top_idx = np.argsort(-scores, kind="mergesort")[:top_k]
        ic_values.append(_spearman_corr_fast(scores, realized))
        hit_values.append(float(np.mean(targets[top_idx])))
    return (
        float(np.nanmean(ic_values)) if len(ic_values) else np.nan,
        float(np.nanmean(hit_values)) if len(hit_values) else np.nan,
    )


def bootstrap_holdout_row_metrics(score_frame, n_iterations=BOOTSTRAP_ITERATIONS):
    if n_iterations <= 0 or len(score_frame) == 0:
        return pd.DataFrame()
    rng = np.random.default_rng(SEED)
    rows = []
    needed = ["Model", "month", "target_top_k_next_1m", "score", "forward_1m_return"]
    compact = score_frame[needed].copy()
    compact["target_top_k_next_1m"] = compact["target_top_k_next_1m"].astype(np.int8)
    compact["score"] = pd.to_numeric(compact["score"], errors="coerce").astype(np.float32)
    compact["forward_1m_return"] = pd.to_numeric(compact["forward_1m_return"], errors="coerce").astype(np.float32)
    for model_name, model_group in tqdm(list(compact.groupby("Model", sort=False)), desc="Bootstrap row metrics", unit="model"):
        month_values = model_group["month"].astype(str).to_numpy()
        unique_months = pd.unique(month_values)
        month_to_code = {month: idx for idx, month in enumerate(unique_months)}
        month_codes = np.asarray([month_to_code[month] for month in month_values], dtype=np.int16)
        month_indices = [np.flatnonzero(month_codes == idx) for idx in range(len(unique_months))]
        y_values = model_group["target_top_k_next_1m"].to_numpy(dtype=np.int8, copy=False)
        score_values = model_group["score"].to_numpy(dtype=np.float32, copy=False)
        return_values = model_group["forward_1m_return"].to_numpy(dtype=np.float32, copy=False)
        metric_values = {"average_precision": [], "mean_spearman_ic": [], "top_k_hit_rate": []}
        for _ in range(n_iterations):
            sampled_month_codes = rng.integers(0, len(unique_months), size=len(unique_months))
            sampled_idx = np.concatenate([month_indices[code] for code in sampled_month_codes])
            if sampled_idx.size == 0:
                continue
            y_sample = y_values[sampled_idx]
            if np.unique(y_sample).size < 2:
                continue
            score_sample = score_values[sampled_idx]
            return_sample = return_values[sampled_idx]
            block_codes = np.concatenate([np.repeat(block_id, len(month_indices[code])) for block_id, code in enumerate(sampled_month_codes)]).astype(np.int16)
            metric_values["average_precision"].append(float(average_precision_score(y_sample, score_sample)))
            mean_ic, mean_hit = _monthly_rank_arrays(y_sample, score_sample, return_sample, block_codes, top_k=TARGET_TOP_K)
            metric_values["mean_spearman_ic"].append(mean_ic)
            metric_values["top_k_hit_rate"].append(mean_hit)
        point_monthly = monthly_rank_quality(model_group.assign(**{"Evaluation Window": "holdout"}), top_k=TARGET_TOP_K)
        point_estimates = {
            "average_precision": safe_metric(average_precision_score, y_values, score_values),
            "mean_spearman_ic": point_monthly["spearman_ic"].mean() if len(point_monthly) else np.nan,
            "top_k_hit_rate": point_monthly["top_k_hit_rate"].mean() if len(point_monthly) else np.nan,
        }
        for metric_name, values in metric_values.items():
            lower, median, upper = bootstrap_quantiles(values)
            rows.append({"Model": model_name, "Metric": metric_name, "Point Estimate": point_estimates.get(metric_name, np.nan), "Bootstrap Median": median, "CI Lower": lower, "CI Upper": upper, "Bootstrap Iterations Requested": n_iterations, "Bootstrap Iterations Used": len([value for value in values if np.isfinite(value)]), "Confidence Level": BOOTSTRAP_CONFIDENCE_LEVEL})
    return pd.DataFrame(rows)

publication_bootstrap_uncertainty = bootstrap_holdout_row_metrics(selected_holdout_predictions, n_iterations=BOOTSTRAP_ITERATIONS)
if len(publication_bootstrap_uncertainty) > 0:
    display(publication_bootstrap_uncertainty.round(4))
    save_artifact_table(publication_bootstrap_uncertainty, "publication_month_block_bootstrap_uncertainty.csv")
cleanup_runtime_memory("after_row_bootstrap")

if len(allocation_summary) > 0:
    save_text_artifact("publication_holdout_summary.txt", allocation_summary.loc[allocation_summary["Evaluation Window"] == "holdout", performance_columns].round(6).to_string(index=False), description="Plain-text holdout model summary.")
if len(portfolio_summary) > 0:
    save_text_artifact("publication_portfolio_summary.txt", portfolio_summary.round(6).to_string(index=False), description="Plain-text portfolio diagnostic summary.")
if len(monthly_rank_aggregate) > 0:
    save_text_artifact("publication_monthly_rank_aggregate.txt", monthly_rank_aggregate.round(6).to_string(index=False), description="Plain-text monthly ranking diagnostics.")


## 7. Drift, Leakage, and Reuse Checklist

This checklist records checks that can be performed with public data. It does not certify live tradability or point-in-time vendor correctness. A production workflow would need stricter data lineage, corporate-action, execution-cost, tax, liquidity, and mandate-constraint reviews.

In [ ]:
def population_stability_index(expected, actual, n_bins=10):
    expected = pd.Series(expected).replace([np.inf, -np.inf], np.nan).dropna()
    actual = pd.Series(actual).replace([np.inf, -np.inf], np.nan).dropna()
    if len(expected) < n_bins or len(actual) < n_bins:
        return np.nan
    quantiles = np.unique(np.quantile(expected, np.linspace(0.0, 1.0, n_bins + 1)))
    if len(quantiles) <= 2:
        return np.nan
    expected_bins = pd.cut(expected, bins=quantiles, include_lowest=True, duplicates="drop")
    actual_bins = pd.cut(actual, bins=quantiles, include_lowest=True, duplicates="drop")
    expected_pct = expected_bins.value_counts(normalize=True, sort=False).replace(0.0, 1e-6)
    actual_pct = actual_bins.value_counts(normalize=True, sort=False).reindex(expected_pct.index).fillna(1e-6).replace(0.0, 1e-6)
    return float(((actual_pct - expected_pct) * np.log(actual_pct / expected_pct)).sum())

feature_drift_rows = []
for column in tqdm(X_selection.columns, desc="Computing feature drift PSI", unit="feature"):
    selection_values = X_selection[column]
    holdout_values = X_holdout[column]
    feature_drift_rows.append(
        {
            "Feature": column,
            "Selection Mean": selection_values.mean(),
            "Holdout Mean": holdout_values.mean(),
            "Selection Std": selection_values.std(),
            "Holdout Std": holdout_values.std(),
            "PSI": population_stability_index(selection_values, holdout_values, n_bins=10),
        }
    )
feature_drift_summary = pd.DataFrame(feature_drift_rows).sort_values("PSI", ascending=False, na_position="last").reset_index(drop=True)
display(feature_drift_summary.head(20).round(4))
save_artifact_table(feature_drift_summary, "feature_drift_summary.csv")
cleanup_runtime_memory("after_feature_drift")
save_text_artifact("publication_feature_drift_summary.txt", feature_drift_summary.head(30).round(6).to_string(index=False), description="Plain-text top feature drift diagnostics.")

chronological_split_ordered = (
    selection_df["date"].max() < calibration_df["date"].min()
    and calibration_df["date"].max() < holdout_df["date"].min()
)
feature_name_leakage_flags = [column for column in X_selection.columns if column.startswith("target_") or column.startswith("forward_") or "next_" in column.lower()]
price_window_has_forward_gap = holdout_df["date"].max() < model_frame["date"].max() or model_frame["forward_1m_return"].notna().all()

if "fred_series_availability_summary" in globals() and len(fred_series_availability_summary) > 0:
    fred_included_count = int(fred_series_availability_summary["include_in_features"].sum())
    fred_excluded_count = int((~fred_series_availability_summary["include_in_features"]).sum())
    fred_excluded_names = fred_series_availability_summary.loc[~fred_series_availability_summary["include_in_features"], "series_id"].tolist()
    fred_availability_status = "review" if fred_excluded_count else "pass"
    fred_availability_evidence = f"included={fred_included_count}; excluded={fred_excluded_count}; excluded_series={fred_excluded_names}"
else:
    fred_included_count = 0
    fred_excluded_count = 0
    fred_availability_status = "not_available"
    fred_availability_evidence = "fred_series_availability_summary was not available"

leakage_checks = pd.DataFrame(
    [
        {"Check": "Target columns excluded from model features", "Status": "pass" if not feature_name_leakage_flags else "fail", "Evidence": f"flagged_feature_columns={feature_name_leakage_flags[:10]}; final_feature_count={X_selection.shape[1]}"},
        {"Check": "Chronological split order", "Status": "pass" if chronological_split_ordered else "fail", "Evidence": f"selection_max={selection_df['date'].max().date()}, calibration_min={calibration_df['date'].min().date()}, calibration_max={calibration_df['date'].max().date()}, holdout_min={holdout_df['date'].min().date()}"},
        {"Check": "Feature policy fitted before calibration and holdout", "Status": "pass", "Evidence": f"feature medians and retained columns fitted on selection window through {TUNING_END_DATE}"},
        {"Check": "Next-month target uses future prices only as label", "Status": "pass", "Evidence": "forward_1m_return and target_top_k_next_1m are excluded from feature columns and used only for training/evaluation labels"},
        {"Check": "Month-end signal and next-month execution convention", "Status": "documented_assumption", "Evidence": f"features use month-end close observations; selected execution_return_mode={EXECUTION_RETURN_MODE}; close-to-close and next-open returns are both saved for audit"},
        {"Check": "Static identity feature policy", "Status": "pass" if FEATURE_SET_VARIANT in {"ticker_ablated", "identity_ablated"} else "documented_assumption", "Evidence": f"feature_set_variant={FEATURE_SET_VARIANT}; excluded_variant_feature_columns={len(excluded_variant_feature_columns)}"},
        {"Check": "FRED series feature availability", "Status": fred_availability_status, "Evidence": fred_availability_evidence},
        {"Check": "Portfolio top-k policy", "Status": "review", "Evidence": "all score-driven portfolios use fixed top-k selection; no holdout optimization of top-k or transaction cost is performed"},
        {"Check": "Transaction costs included", "Status": "pass", "Evidence": f"transaction_cost_bps={TRANSACTION_COST_BPS}; cost applied to one-way monthly turnover"},
        {"Check": "Turnover-cap sensitivity", "Status": "pass" if len(TURNOVER_CONSTRAINT_CAPS) > 0 else "not_requested", "Evidence": f"turnover_constraint_caps={TURNOVER_CONSTRAINT_CAPS}"},
        {"Check": "Public market data point-in-time limitations", "Status": "known_limitation", "Evidence": "yfinance/FRED/Cboe public files are reproducible but not equivalent to a licensed point-in-time production data store"},
        {"Check": "Investment interpretation", "Status": "educational_only", "Evidence": "portfolio section is a diagnostic of score utility, not an investment recommendation"},
    ]
)
display(leakage_checks)
save_artifact_table(leakage_checks, "leakage_checks.csv")

if cuda_memory_snapshots:
    cuda_memory_summary = pd.concat(cuda_memory_snapshots, ignore_index=True)
    save_artifact_table(cuda_memory_summary, "cuda_memory_summary_final.csv")

registered_artifact_manifest = pd.DataFrame(artifact_manifest_rows).drop_duplicates(subset=["path", "kind"], keep="last")
if len(registered_artifact_manifest) > 0:
    display(registered_artifact_manifest)
    save_artifact_table(registered_artifact_manifest, "registered_artifact_manifest.csv")

run_closeout = f"""
# Run Closeout

Artifact directory: {ARTIFACT_DIR}
Run mode: {'FAST_MODE smoke test' if FAST_MODE else 'full research run'}
Rows in model frame: {len(model_frame):,}
Months in model frame: {model_frame['month'].nunique():,}
Asset count: {model_frame['asset'].nunique():,}
Feature count: {X_selection.shape[1]:,}
Feature set variant: {FEATURE_SET_VARIANT}
Execution return mode: {EXECUTION_RETURN_MODE}
FRED series included in features: {fred_included_count:,}
FRED series excluded for insufficient selection history or load failure: {fred_excluded_count:,}
Holdout rows: {len(holdout_df):,}
Holdout months: {holdout_df['month'].nunique():,}
Holdout top-k label rate: {y_holdout.mean():.4%}
Successful model rows: {len(allocation_summary):,}
Model errors: {len(model_errors):,}

Use the CSV/TXT/PNG artifacts for offline review because notebook editor output can be truncated.
""".strip()
save_text_artifact("run_closeout.md", run_closeout, description="Final run-level summary and artifact guidance.")


## 8. Save Output Archive

The final cell writes a manifest and creates a ZIP archive for download from Kaggle.

In [ ]:
artifact_file_rows = []
for artifact_path in tqdm(sorted(ARTIFACT_DIR.glob("**/*")), desc="Building artifact manifest", unit="path"):
    if artifact_path.is_file():
        artifact_file_rows.append(
            {
                "path": str(artifact_path),
                "filename": artifact_path.name,
                "suffix": artifact_path.suffix,
                "size_bytes": artifact_path.stat().st_size,
            }
        )

artifact_file_manifest = pd.DataFrame(artifact_file_rows)
if len(artifact_file_manifest) > 0:
    display(artifact_file_manifest)
    save_artifact_table(artifact_file_manifest, "artifact_file_manifest.csv", description="Files present in the artifact directory at archive time.")

import shutil

archive_base_candidates = [
    Path("/kaggle/working/tabpfn_tabicl_tactical_asset_allocation_20260511_outputs"),
    ARTIFACT_DIR.parent / "tabpfn_tabicl_tactical_asset_allocation_20260511_outputs",
]
for archive_base in archive_base_candidates:
    try:
        archive_base.parent.mkdir(parents=True, exist_ok=True)
        archive_path = shutil.make_archive(str(archive_base), "zip", ARTIFACT_DIR)
        print(f"Created {archive_path}")
        break
    except Exception as exc:
        print(f"Could not create archive at {archive_base}.zip: {short_error(exc)}")

print("Notebook workflow complete. Review CSV, TXT, and PNG artifacts for the run record.")
